In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Pengambilan Data

In [ ]:
import pandas as pd

# Definisikan path file
path_sent = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'
path_hist = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'

# Baca 5 baris pertama
df_sent = pd.read_csv(path_sent, encoding='latin1', low_memory=False, nrows=5)
df_hist = pd.read_csv(path_hist, low_memory=False, nrows=5)

In [ ]:
# Tampilkan
print("=== 5 baris pertama Twitter Sentiment ===")
print(df_sent)

In [ ]:
print("\n=== 5 baris pertama Data Historis Bitcoin ===")
print(df_hist)

In [ ]:
import os, glob
import pandas as pd

# 1. List direktori yang mungkin
search_paths = ['/kaggle/input', '/mnt/data', os.getcwd()]

# 2. Cari CSV historis dan sentiment
hist_files = []
sent_files = []
for base in search_paths:
    hist_files += glob.glob(os.path.join(base, '', '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'), recursive=True)
    sent_files += glob.glob(os.path.join(base, '', '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'), recursive=True)

if not hist_files or not sent_files:
    raise FileNotFoundError(
        f"File historis ditemukan: {hist_files}\n"
        f"File sentiment ditemukan: {sent_files}\n"
        "Pastikan CSV sudah tersedia di salah satu direktori di atas."
    )

hist_path = hist_files[0]
sent_path = sent_files[0]

print("Menggunakan file historis :", hist_path)
print("Menggunakan file sentiment:", sent_path)

# 3. Load data tanpa modifikasi struktur
df_hist = pd.read_csv(hist_path, low_memory=False)
df_sent = pd.read_csv(sent_path, encoding='latin1', low_memory=False, on_bad_lines='skip')

# 4. Preview
print("\n=== Historical Data ===")
print(df_hist.shape)
print(df_hist.columns.tolist())
print(df_hist.head())

print("\n=== Twitter Sentiment Data ===")
print(df_sent.shape)
print(df_sent.columns.tolist())
print(df_sent.head())

In [ ]:
import pandas as pd

# 1. Load CSV (as sebelumnya)
hist_path = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'

df_hist = pd.read_csv(hist_path, low_memory=False)
df_sent = pd.read_csv(sent_path, encoding='latin1', low_memory=False, on_bad_lines='skip')

# 2. Tambah kolom datetime dan set index
df_hist['datetime'] = pd.to_datetime(df_hist['Timestamp'], unit='s', errors='coerce')
df_sent['datetime'] = pd.to_datetime(df_sent['date'], errors='coerce')

df_hist.set_index('datetime', inplace=True)
df_sent.set_index('datetime', inplace=True)

# 3. Drop duplikat di index (keep pertama)
df_hist_u = df_hist[~df_hist.index.duplicated(keep='first')]
df_sent_u = df_sent[~df_sent.index.duplicated(keep='first')]

# 4. Merge inner
df_merged = df_hist_u.join(df_sent_u, how='inner', lsuffix='_hist', rsuffix='_twit')

print("Merged shape:", df_merged.shape)
print(df_merged.head())

In [ ]:
import pandas as pd

# Paths ke CSV
hist_path = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'

# 1. Load data
df_hist = pd.read_csv(hist_path, low_memory=False)
df_sent = pd.read_csv(sent_path, encoding='latin1', low_memory=False, on_bad_lines='skip')

# 2. Buat datetime index
df_hist['datetime'] = pd.to_datetime(df_hist['Timestamp'], unit='s', errors='coerce')
df_sent['datetime'] = pd.to_datetime(df_sent['date'], errors='coerce')
df_hist = df_hist.set_index('datetime').sort_index()
df_sent = df_sent.set_index('datetime').sort_index()

# 3. Drop NaT dan pastikan sorted
df_hist = df_hist[~df_hist.index.isna()]
df_sent = df_sent[~df_sent.index.isna()]

# 4. Merge_asof dengan toleransi ±30 detik
df_min = pd.merge_asof(
    df_hist, 
    df_sent[['Polarity Score']].rename(columns={'Polarity Score':'sentiment'}),
    left_index=True, 
    right_index=True,
    direction='nearest',
    tolerance=pd.Timedelta('30s')
)

# 5. Isi NaN sentiment: ffill/bfill (limit 60), lalu 0
df_min['sentiment'] = (
    df_min['sentiment']
    .ffill(limit=60)
    .bfill(limit=60)
    .fillna(0.0)
)

# 6. Inspect hasil
print("Merged-asof shape with sentiment fill:", df_min.shape)
print(df_min[['Open','Close','sentiment']].head(10))

In [ ]:
import pandas as pd

# ——— Sesuaikan dua baris ini dengan path CSV-mu ——————————————
hist_path = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'
# ————————————————————————————————————————————————

# 1. Load data
df_hist = pd.read_csv(hist_path, low_memory=False)
df_sent = pd.read_csv(sent_path, encoding='latin1', low_memory=False, on_bad_lines='skip')

# 2. Buat datetime index
df_hist['datetime'] = pd.to_datetime(df_hist['Timestamp'], unit='s', errors='coerce')
df_sent['datetime'] = pd.to_datetime(df_sent['date'], errors='coerce')
df_hist.set_index('datetime', inplace=True)
df_sent.set_index('datetime', inplace=True)

# 3. Drop baris tanpa datetime
df_hist = df_hist[df_hist.index.notna()]
df_sent = df_sent[df_sent.index.notna()]

# 4. Merge_asof ±30 detik
df_min = pd.merge_asof(
    df_hist.sort_index(),
    df_sent[['Polarity Score']].rename(columns={'Polarity Score':'sentiment'}).sort_index(),
    left_index=True,
    right_index=True,
    direction='nearest',
    tolerance=pd.Timedelta('30s')
)

# 5. Isi NaN sentiment (ffill/bfill hingga 60 baris, sisanya = 0)
df_min['sentiment'] = (
    df_min['sentiment']
    .ffill(limit=60)
    .bfill(limit=60)
    .fillna(0.0)
)

# 6. Resample ke harian OHLCV + rata-rata sentiment
df_daily = df_min.resample('D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum',
    'sentiment': 'mean'
})

# 7. Tambahkan indikator teknikal:
df_daily['SMA_7'] = df_daily['Close'].rolling(7, min_periods=1).mean()
df_daily['EMA_7'] = df_daily['Close'].ewm(span=7, adjust=False).mean()
delta = df_daily['Close'].diff()
up = delta.clip(lower=0).ewm(com=13, adjust=False).mean()
down = (-delta.clip(upper=0)).ewm(com=13, adjust=False).mean()
df_daily['RSI_14'] = 100 - (100/(1 + up/down))

# 8. Drop hari tanpa data harga
df_daily.dropna(subset=['Open','High','Low','Close','Volume'], inplace=True)

# 9. Tampilkan hasil
print("Dataset harian dengan fitur shape:", df_daily.shape)
df_daily.head(10)

# Preprocessing & Resampling

> Usecols & parse_dates

In [ ]:
import pandas as pd

hist_path = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'

# 1. Load data harga per‐menit
df_hist = pd.read_csv(
    hist_path,
    usecols=['Timestamp','Open','High','Low','Close','Volume'],
    dtype={
        'Timestamp': 'int64',
        'Open':      'float32',
        'High':      'float32',
        'Low':       'float32',
        'Close':     'float32',
        'Volume':    'float32',
    },
    low_memory=False
)
df_hist['datetime'] = pd.to_datetime(df_hist['Timestamp'], unit='s', errors='coerce')
df_hist = (
    df_hist
    .drop(columns='Timestamp')
    .set_index('datetime')
    .sort_index()
)


# 2. Load data sentiment Twitter
#    – gunakan default C engine (hapus engine='python')
#    – encoding latin1, skip baris korup, low_memory=False
df_sent = pd.read_csv(
    sent_path,
    usecols=['date','Polarity Score'],
    dtype={'Polarity Score':'float32'},
    encoding='latin1',
    on_bad_lines='skip',
    low_memory=False
)
df_sent['datetime'] = pd.to_datetime(df_sent['date'], errors='coerce')
df_sent = (
    df_sent
    .drop(columns='date')
    .rename(columns={'Polarity Score':'sentiment'})
    .set_index('datetime')
    .sort_index()
)

# Preview
print(df_hist.head())
print(df_sent.head())


> Chunking untuk tweet

In [ ]:
import pandas as pd

# path ke file sentiment (sesuaikan jika berbeda)
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'

usecols_sent = ['date', 'Polarity Score']
dtypes_sent  = {'Polarity Score': 'float32'}
chunksize    = 200_000

chunks = []
for chunk in pd.read_csv(
    sent_path,
    usecols=usecols_sent,
    dtype=dtypes_sent,
    encoding='latin1',
    on_bad_lines='skip',
    chunksize=chunksize
):
    # parse datetime, hapus kolom lama, rename, index, sort
    chunk['datetime'] = pd.to_datetime(chunk['date'], errors='coerce')
    chunk.drop(columns='date', inplace=True)
    chunk.rename(columns={'Polarity Score': 'sentiment'}, inplace=True)
    chunk.set_index('datetime', inplace=True)
    chunk.sort_index(inplace=True)
    chunks.append(chunk)

# gabungkan semua chunk dan hilangkan index duplikat
df_sent = pd.concat(chunks, ignore_index=False)
df_sent = df_sent[~df_sent.index.duplicated(keep='first')]

print(df_sent.head())


> Indexing & sorting

In [ ]:
# Pastikan df_hist sudah diload sebelumnya, misal:
# df_hist = pd.read_csv(..., usecols=[...], parse_dates=[...], ...)
# Jika kamu sudah drop kolom 'Timestamp' dan rename jadi index 'datetime',
# skip bagian convert ulang dan langsung ke sort/drop.

import pandas as pd

# --- df_hist: indexing & sorting ---
# Cek apakah index sudah datetime
if not isinstance(df_hist.index, pd.DatetimeIndex):
    if 'datetime' in df_hist.columns:
        df_hist = df_hist.set_index('datetime')
    elif 'Timestamp' in df_hist.columns:
        # Jika Timestamp masih ada, convert
        df_hist['datetime'] = pd.to_datetime(df_hist['Timestamp'], unit='s', errors='coerce')
        df_hist = df_hist.drop(columns='Timestamp').set_index('datetime')
df_hist = df_hist.sort_index()
df_hist = df_hist[~df_hist.index.isna()]               # drop NaT
df_hist = df_hist[~df_hist.index.duplicated(keep='first')]  # drop duplikat

# --- df_sent: indexing & sorting ---
if not isinstance(df_sent.index, pd.DatetimeIndex):
    if 'datetime' in df_sent.columns:
        df_sent = df_sent.set_index('datetime')
    elif 'date' in df_sent.columns:
        df_sent['datetime'] = pd.to_datetime(df_sent['date'], errors='coerce')
        df_sent = (
            df_sent
            .drop(columns='date')
            .set_index('datetime')
        )
df_sent = df_sent.sort_index()
df_sent = df_sent[~df_sent.index.isna()]
df_sent = df_sent[~df_sent.index.duplicated(keep='first')]

# Preview
print(df_hist.head())
print(df_sent.head())


> Merge_asof

In [ ]:
import pandas as pd

# — pastikan df_hist dan df_sent sudah ada (dari load & indexing sebelumnya) —

# 1) Merge per‐menit
df_min = pd.merge_asof(
    df_hist,
    df_sent[['sentiment']],
    left_index=True,
    right_index=True,
    direction='nearest',
    tolerance=pd.Timedelta('30s')
)

# 2) Fill missing sentiment
df_min['sentiment'] = (
    df_min['sentiment']
      .ffill(limit=60)
      .bfill(limit=60)
      .fillna(0.0)
)

# 3) (Opsional) Simpan intermediate
df_min.to_parquet('df_minute.parquet')

# 4) Resample harian
df_daily = pd.DataFrame({
    'Open'     : df_min['Open'].resample('D').first(),
    'High'     : df_min['High'].resample('D').max(),
    'Low'      : df_min['Low'].resample('D').min(),
    'Close'    : df_min['Close'].resample('D').last(),
    'Volume'   : df_min['Volume'].resample('D').sum(),
    'sentiment': df_min['sentiment'].resample('D').mean(),
})
df_daily.dropna(subset=['Open'], inplace=True)

# 5) Preview
print(df_min[['Open','Close','sentiment']].head())
print(df_daily.head())


> Fill missing sentiment

In [ ]:
# Fill missing sentiment in df_min (forward-fill & backward-fill up to 60 rows, then fill remaining NaN with 0)
df_min['sentiment'] = (
    df_min['sentiment']
      .ffill(limit=60)
      .bfill(limit=60)
      .fillna(0.0)
)

# Preview to ensure fill worked
print(df_min[['Open', 'Close', 'sentiment']].head(10))

> Outlier detection (opsional)

In [ ]:
# 1) Merge per‐menit
df_min = pd.merge_asof(
    df_hist,
    df_sent[['sentiment']],
    left_index=True,
    right_index=True,
    direction='nearest',
    tolerance=pd.Timedelta('30s')
)

# 2) Fill missing sentiment
df_min['sentiment'] = df_min['sentiment'].ffill(limit=60).bfill(limit=60).fillna(0.0)


In [ ]:
import numpy as np

# Outlier harga
mean_close = df_min['Close'].mean()
std_close  = df_min['Close'].std()
df_min['zscore_close']      = (df_min['Close'] - mean_close) / std_close
df_min['outlier_z_close']   = df_min['zscore_close'].abs() > 3
Q1 = df_min['Close'].quantile(0.25)
Q3 = df_min['Close'].quantile(0.75)
IQR = Q3 - Q1
df_min['outlier_iqr_close'] = (df_min['Close'] < Q1 - 1.5*IQR) | (df_min['Close'] > Q3 + 1.5*IQR)

# Outlier sentiment
mean_sent = df_min['sentiment'].mean()
std_sent  = df_min['sentiment'].std()
df_min['zscore_sent']      = (df_min['sentiment'] - mean_sent) / std_sent
df_min['outlier_z_sent']   = df_min['zscore_sent'].abs() > 3
Q1s = df_min['sentiment'].quantile(0.25)
Q3s = df_min['sentiment'].quantile(0.75)
IQRs = Q3s - Q1s
df_min['outlier_iqr_sent'] = (df_min['sentiment'] < Q1s - 1.5*IQRs) | (df_min['sentiment'] > Q3s + 1.5*IQRs)

print("Outliers (Price) by Z-score:", df_min['outlier_z_close'].sum())
print("Outliers (Price) by IQR:", df_min['outlier_iqr_close'].sum())
print("Outliers (Sentiment) by Z-score:", df_min['outlier_z_sent'].sum())
print("Outliers (Sentiment) by IQR:", df_min['outlier_iqr_sent'].sum())


> Checkpoint intermediate

In [ ]:
# Checkpoint Intermediate: Save df_min to disk

# Simpan df_min sebagai Parquet
df_min.to_parquet('df_minute.parquet')

# Opsional: simpan sebagai CSV
df_min.to_csv('df_minute.csv')

# Konfirmasi file tersimpan
import os
print("Files in working directory:", os.listdir())

> Resample harian (OHLCV + sentiment mean)

In [ ]:
# Resample harian (OHLCV + rata‐rata sentiment)
df_daily = pd.DataFrame({
    'Open'     : df_min['Open'].resample('D').first(),
    'High'     : df_min['High'].resample('D').max(),
    'Low'      : df_min['Low'].resample('D').min(),
    'Close'    : df_min['Close'].resample('D').last(),
    'Volume'   : df_min['Volume'].resample('D').sum(),
    'sentiment': df_min['sentiment'].resample('D').mean(),
})
# Hapus hari tanpa data harga
df_daily.dropna(subset=['Open'], inplace=True)

# Preview hasil
print(df_daily.head())

> Drop hari tanpa data

In [ ]:
# Resample harian (jika belum)
df_daily = pd.DataFrame({
    'Open'     : df_min['Open'].resample('D').first(),
    'High'     : df_min['High'].resample('D').max(),
    'Low'      : df_min['Low'].resample('D').min(),
    'Close'    : df_min['Close'].resample('D').last(),
    'Volume'   : df_min['Volume'].resample('D').sum(),
    'sentiment': df_min['sentiment'].resample('D').mean(),
})


In [ ]:
# Drop hari tanpa data harga
df_daily = df_daily.dropna(subset=['Open'])

# Preview
print("Jumlah hari setelah drop:", len(df_daily))
print(df_daily.head())

> Modularisasi

In [ ]:
# Update preprocess_merge_asof to drop NaN-index and sort both sides before merge_asof

def preprocess_merge_asof(df_hist, df_sent, tol='30s', fill_limit=60):
    # ensure hist and sent are sorted
    df_hist = df_hist.sort_index()
    
    # drop null-index rows and sort sentiment
    df_sent = df_sent[~df_sent.index.isna()]
    df_sent = df_sent.sort_index()

    df_min = pd.merge_asof(
        df_hist,
        df_sent[['sentiment']],
        left_index=True,
        right_index=True,
        direction='nearest',
        tolerance=pd.Timedelta(tol)
    )
    df_min['sentiment'] = (
        df_min['sentiment']
          .ffill(limit=fill_limit)
          .bfill(limit=fill_limit)
          .fillna(0.0)
    )
    return df_min

# Usage remains the same:
# df_min = preprocess_merge_asof(df_hist, df_sent)

In [ ]:
def preprocess_merge_asof(df_hist, df_sent, tol='30s', fill_limit=60):
    # drop any null‐datetime rows in sentiment
    df_sent = df_sent[~df_sent.index.isna()]

    df_min = pd.merge_asof(
        df_hist,
        df_sent[['sentiment']],
        left_index=True,
        right_index=True,
        direction='nearest',
        tolerance=pd.Timedelta(tol)
    )
    df_min['sentiment'] = (
        df_min['sentiment']
          .ffill(limit=fill_limit)
          .bfill(limit=fill_limit)
          .fillna(0.0)
    )
    return df_min


In [ ]:
import pandas as pd

def load_hist_data(path):
    df = pd.read_csv(
        path,
        usecols=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'],
        dtype={
            'Timestamp': 'int64',
            'Open':      'float32',
            'High':      'float32',
            'Low':       'float32',
            'Close':     'float32',
            'Volume':    'float32',
        },
        low_memory=False
    )
    df['datetime'] = pd.to_datetime(df['Timestamp'], unit='s', errors='coerce')
    df = df.drop(columns='Timestamp') \
           .set_index('datetime') \
           .sort_index()
    df = df[~df.index.isna()]
    df = df[~df.index.duplicated(keep='first')]
    return df

def load_sent_data(path, chunksize=200_000):
    chunks = []
    for chunk in pd.read_csv(
        path,
        usecols=['date', 'Polarity Score'],
        dtype={'Polarity Score': 'float32'},
        encoding='latin1',
        on_bad_lines='skip',
        chunksize=chunksize
    ):
        chunk['datetime'] = pd.to_datetime(chunk['date'], errors='coerce')
        chunk = chunk.drop(columns='date') \
                     .rename(columns={'Polarity Score': 'sentiment'})
        chunk = chunk.set_index('datetime').sort_index()
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=False)
    df = df[~df.index.duplicated(keep='first')]
    return df

def preprocess_merge_asof(df_hist, df_sent, tol='30s', fill_limit=60):
    df_hist = df_hist.sort_index()
    df_sent = df_sent[~df_sent.index.isna()].sort_index()
    df_min = pd.merge_asof(
        df_hist,
        df_sent[['sentiment']],
        left_index=True,
        right_index=True,
        direction='nearest',
        tolerance=pd.Timedelta(tol)
    )
    df_min['sentiment'] = (
        df_min['sentiment']
          .ffill(limit=fill_limit)
          .bfill(limit=fill_limit)
          .fillna(0.0)
    )
    return df_min

def detect_outliers(df_min):
    # Price outliers
    mc, sc = df_min['Close'].mean(), df_min['Close'].std()
    df_min['zscore_close']    = (df_min['Close'] - mc) / sc
    df_min['outlier_z_close'] = df_min['zscore_close'].abs() > 3
    q1, q3 = df_min['Close'].quantile([0.25, 0.75])
    iqr = q3 - q1
    df_min['outlier_iqr_close'] = (df_min['Close'] < q1 - 1.5*iqr) | (df_min['Close'] > q3 + 1.5*iqr)
    # Sentiment outliers
    ms, ss = df_min['sentiment'].mean(), df_min['sentiment'].std()
    df_min['zscore_sent']    = (df_min['sentiment'] - ms) / ss
    df_min['outlier_z_sent'] = df_min['zscore_sent'].abs() > 3
    q1s, q3s = df_min['sentiment'].quantile([0.25, 0.75])
    iqrs = q3s - q1s
    df_min['outlier_iqr_sent'] = (df_min['sentiment'] < q1s - 1.5*iqrs) | (df_min['sentiment'] > q3s + 1.5*iqrs)
    return df_min

def save_checkpoint(df, filename):
    df.to_parquet(filename)

def resample_daily(df_min):
    df_daily = pd.DataFrame({
        'Open'     : df_min['Open'].resample('D').first(),
        'High'     : df_min['High'].resample('D').max(),
        'Low'      : df_min['Low'].resample('D').min(),
        'Close'    : df_min['Close'].resample('D').last(),
        'Volume'   : df_min['Volume'].resample('D').sum(),
        'sentiment': df_min['sentiment'].resample('D').mean(),
    })
    df_daily.dropna(subset=['Open'], inplace=True)
    return df_daily

# Example usage in Kaggle environment
hist_path = '/kaggle/input/bitcoin-historical-data/btcusd_1-min_data.csv'
sent_path = '/kaggle/input/bitcoin-sentiment-analysis-twitter-data/bitcoin_tweets1000000.csv'

df_hist = load_hist_data(hist_path)
df_sent = load_sent_data(sent_path)
df_min  = preprocess_merge_asof(df_hist, df_sent)
df_min  = detect_outliers(df_min)
save_checkpoint(df_min, 'df_minute.parquet')
df_daily = resample_daily(df_min)
save_checkpoint(df_daily, 'df_daily.parquet')

print("Pipeline complete: df_minute.parquet & df_daily.parquet created")


# Feature Engineering

> Indikator Teknikal (Harga)

In [ ]:
import warnings

# Suppress runtime warnings (na comparisons)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# — Hitung indikator teknikal —

# 1) Simple Moving Averages
df_daily['SMA7']  = df_daily['Close'].rolling(window=7,  min_periods=7).mean()
df_daily['SMA14'] = df_daily['Close'].rolling(window=14, min_periods=14).mean()
df_daily['SMA21'] = df_daily['Close'].rolling(window=21, min_periods=21).mean()

# 2) Exponential Moving Averages
df_daily['EMA7']  = df_daily['Close'].ewm(span=7,  adjust=False).mean()
df_daily['EMA14'] = df_daily['Close'].ewm(span=14, adjust=False).mean()

# 3) Relative Strength Index (14 hari)
delta    = df_daily['Close'].diff()
gain     = delta.clip(lower=0)
loss     = -delta.clip(upper=0)
avg_gain = gain.rolling(window=14, min_periods=14).mean()
avg_loss = loss.rolling(window=14, min_periods=14).mean()
rs       = avg_gain / avg_loss
df_daily['RSI14'] = 100 - (100 / (1 + rs))

# 4) MACD (12–26) dan Signal line (9)
ema12 = df_daily['Close'].ewm(span=12, adjust=False).mean()
ema26 = df_daily['Close'].ewm(span=26, adjust=False).mean()
df_daily['MACD']        = ema12 - ema26
df_daily['MACD_Signal'] = df_daily['MACD'].ewm(span=9, adjust=False).mean()

# 5) Bollinger Bands (20 hari ±2σ)
bb_mid = df_daily['Close'].rolling(window=20, min_periods=20).mean()
bb_std = df_daily['Close'].rolling(window=20, min_periods=20).std()
df_daily['BB_Upper'] = bb_mid + 2 * bb_std
df_daily['BB_Lower'] = bb_mid - 2 * bb_std

# 6) Buang baris awal yang belum lengkap
df_daily = df_daily.dropna(subset=['SMA7','RSI14','MACD','BB_Upper'])

# Preview
print(df_daily[['SMA7','EMA7','RSI14','MACD','BB_Upper','BB_Lower']].tail())


> Statistik Harian (Harga & Volume)

In [ ]:
import numpy as np

# 1. Return harian dan log‐return
df_daily['Return']    = df_daily['Close'].pct_change()
df_daily['LogReturn'] = np.log(df_daily['Close'] / df_daily['Close'].shift(1))

# 2. Volatilitas return (rolling std)
df_daily['Vol_7d']  = df_daily['Return'].rolling(window=7,  min_periods=7).std()
df_daily['Vol_14d'] = df_daily['Return'].rolling(window=14, min_periods=14).std()
df_daily['Vol_30d'] = df_daily['Return'].rolling(window=30, min_periods=30).std()

# 3. Perubahan volume relatif (vs rata-rata 7 hari)
df_daily['VolChange_7d'] = (
    df_daily['Volume'] 
    / df_daily['Volume'].rolling(window=7, min_periods=7).mean()
)

# 4. Log-transformasi volume
df_daily['LogVolume'] = np.log1p(df_daily['Volume'])

# 5. (Opsional) Drop baris awal dengan NaN
df_daily = df_daily.dropna(subset=['Return','Vol_7d','VolChange_7d'])

# Preview hasil
print(df_daily[['Return','LogReturn','Vol_7d','Vol_14d','Vol_30d','VolChange_7d','LogVolume']].tail())


> Fitur Lag & Rolling Window

In [ ]:
# === Fitur Lag & Rolling Window ===
# Asumsi: df_daily sudah berisi kolom ['Open','High','Low','Close','Volume','sentiment']

# 1) Lag features (1–7 hari)
for lag in range(1, 8):
    df_daily[f'Close_lag_{lag}']     = df_daily['Close'].shift(lag)
    df_daily[f'Volume_lag_{lag}']    = df_daily['Volume'].shift(lag)
    df_daily[f'Sentiment_lag_{lag}'] = df_daily['sentiment'].shift(lag)

# 2) Rolling min/max/sum untuk Close & Volume (periode 7 & 14 hari)
for window in [7, 14]:
    df_daily[f'Close_roll_min_{window}']  = df_daily['Close'].rolling(window=window,  min_periods=window).min()
    df_daily[f'Close_roll_max_{window}']  = df_daily['Close'].rolling(window=window,  min_periods=window).max()
    df_daily[f'Close_roll_sum_{window}']  = df_daily['Close'].rolling(window=window,  min_periods=window).sum()
    df_daily[f'Volume_roll_min_{window}'] = df_daily['Volume'].rolling(window=window, min_periods=window).min()
    df_daily[f'Volume_roll_max_{window}'] = df_daily['Volume'].rolling(window=window, min_periods=window).max()
    df_daily[f'Volume_roll_sum_{window}'] = df_daily['Volume'].rolling(window=window, min_periods=window).sum()

# 3) Rolling mean/median/std untuk sentiment (periode 3 & 7 hari)
for window in [3, 7]:
    df_daily[f'Sentiment_roll_mean_{window}'] = df_daily['sentiment'].rolling(window=window, min_periods=window).mean()
    df_daily[f'Sentiment_roll_med_{window}']  = df_daily['sentiment'].rolling(window=window, min_periods=window).median()
    df_daily[f'Sentiment_roll_std_{window}']  = df_daily['sentiment'].rolling(window=window, min_periods=window).std()

# 4) (Opsional) Hapus baris awal yang masih mengandung NaN akibat shift/rolling
required = [f'Close_lag_{i}' for i in range(1,8)] + [f'Close_roll_min_7', f'Sentiment_roll_mean_3']
df_daily = df_daily.dropna(subset=required)

# 5) Preview beberapa baris terakhir
print(df_daily.tail())

> Fitur Sentimen

In [ ]:
# === Fitur Sentimen ===
# Asumsi: df_daily sudah berisi kolom 'sentiment'

# 1) Lag sentiment untuk 1, 3, dan 7 hari
for lag in [1, 3, 7]:
    df_daily[f'sentiment_lag_{lag}'] = df_daily['sentiment'].shift(lag)

# 2) Rolling volatility sentiment (7 hari)
df_daily['sentiment_volatility_7d'] = (
    df_daily['sentiment']
    .rolling(window=7, min_periods=7)
    .std()
)

# 3) Indikator biner: sentiment positif vs non-positif
df_daily['sentiment_positive'] = (df_daily['sentiment'] > 0).astype(int)
df_daily['sentiment_negative'] = (df_daily['sentiment'] < 0).astype(int)

# 4) (Opsional) Hapus baris awal yang masih mengandung NaN dari shift/rolling
required = [f'sentiment_lag_{lag}' for lag in [1,3,7]] + ['sentiment_volatility_7d']
df_daily = df_daily.dropna(subset=required)

# 5) Preview hasil
print(df_daily[
    ['sentiment',
     'sentiment_lag_1','sentiment_lag_3','sentiment_lag_7',
     'sentiment_volatility_7d',
     'sentiment_positive','sentiment_negative']
].tail())


> Fitur Kalender & Waktu

In [ ]:
# === Fitur Kalender & Waktu pada df_daily ===
# Asumsi: df_daily sudah berupa DataFrame dengan DatetimeIndex

# 1) Hari dalam minggu (0=Senin … 6=Minggu)
df_daily['day_of_week'] = df_daily.index.dayofweek

# 2) Weekend vs Weekday
df_daily['is_weekend'] = (df_daily['day_of_week'] >= 5).astype(int)

# 3) Hari dalam bulan
df_daily['day_of_month'] = df_daily.index.day

# 4) Bulan dan kuartal
df_daily['month']   = df_daily.index.month
df_daily['quarter'] = df_daily.index.quarter

# 5) Awal/akhir bulan dan kuartal
df_daily['is_month_start']   = df_daily.index.is_month_start.astype(int)
df_daily['is_month_end']     = df_daily.index.is_month_end.astype(int)
df_daily['is_quarter_start'] = df_daily.index.is_quarter_start.astype(int)
df_daily['is_quarter_end']   = df_daily.index.is_quarter_end.astype(int)

# 6) (Opsional) Preview untuk verifikasi
print(df_daily[
    ['day_of_week','is_weekend','day_of_month','month',
     'quarter','is_month_start','is_month_end']
].head(10))

> Interaksi Harga–Sentimen

In [ ]:
# === Interaksi Harga–Sentimen ===
# Asumsi: df_daily sudah berisi kolom 'Return', 'Vol_7d', dan 'sentiment'

# 1) Sentiment × Return
df_daily['sent_x_return'] = df_daily['sentiment'] * df_daily['Return']

# 2) Sentiment × Volatilitas (7 hari)
df_daily['sent_x_volatility'] = df_daily['sentiment'] * df_daily['Vol_7d']

# 3) Return per unit Sentiment (hindari div0 dengan epsilon kecil)
eps = 1e-6
df_daily['return_per_sent'] = df_daily['Return'] / (df_daily['sentiment'].abs() + eps)

# 4) Harga relatif terhadap Sentiment (Close / (1+|sentiment|))
df_daily['price_over_sent'] = df_daily['Close'] / (1 + df_daily['sentiment'].abs())

# 5) (Opsional) Drop baris dengan NaN dari fitur interaksi
df_daily = df_daily.dropna(subset=['sent_x_return','sent_x_volatility','return_per_sent','price_over_sent'])

# 6) Preview beberapa baris
print(df_daily[[
    'sentiment','Return','Vol_7d',
    'sent_x_return','sent_x_volatility',
    'return_per_sent','price_over_sent'
]].tail())

> Transformasi Distribusi

In [ ]:
import numpy as np
from sklearn.preprocessing import PowerTransformer
from scipy.stats.mstats import winsorize

# 1) Yeo–Johnson transform pada Return (karena bisa bernilai negatif)
pt = PowerTransformer(method='yeo-johnson', standardize=False)
df_daily['Return_YJ'] = pt.fit_transform(df_daily[['Return']].fillna(0))

# 2) Box–Cox transform pada Volume (pastikan positif dengan +1)
#    jika Volume sudah >0, bisa langsung; kalau ada zero, tambahkan 1
from scipy import stats
df_daily['Volume_boxcox'], _ = stats.boxcox(df_daily['Volume'] + 1)

# 3) Winsorize beberapa fitur untuk kurangi efek outlier ekstrem
#    Batasi 1% teratas & terbawah
df_daily['Close_winsor']  = winsorize(df_daily['Close'], limits=[0.01, 0.01])
df_daily['Return_winsor'] = winsorize(df_daily['Return'].fillna(0), limits=[0.01, 0.01])

# 4) Preview hasil transformasi
print(df_daily[
    ['Return','Return_YJ','Return_winsor',
     'Volume','Volume_boxcox',
     'Close','Close_winsor']
].tail())

# Pelatihan Base Models

> Definisikan Target & Problem

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
from scipy.stats.mstats import winsorize

# 0. Definisikan transformer
class DistributionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,
                 yeojohnson_cols=None,
                 boxcox_cols=None,
                 winsor_cols=None,
                 winsor_limits=(0.01, 0.01)):
        self.yeojohnson_cols = yeojohnson_cols or []
        self.boxcox_cols     = boxcox_cols    or []
        self.winsor_cols     = winsor_cols    or []
        self.winsor_limits   = winsor_limits
        self._pt             = {}
        self._lambdas        = {}

    def fit(self, X, y=None):
        for col in self.yeojohnson_cols:
            pt = PowerTransformer(method='yeo-johnson', standardize=True)
            pt.fit(X[[col]])
            self._pt[col] = pt
        for col in self.boxcox_cols:
            arr = X[col].values + 1.0
            _, lmbda = boxcox(arr)
            self._lambdas[col] = lmbda
        return self

    def transform(self, X):
        X = X.copy()
        # Yeo–Johnson
        for col, pt in self._pt.items():
            X[f"{col}_yj"] = pt.transform(X[[col]])
        # Box–Cox
        for col, lmbda in self._lambdas.items():
            arr = X[col].values + 1.0
            if np.isclose(lmbda, 0):
                X[f"{col}_bc"] = np.log(arr)
            else:
                X[f"{col}_bc"] = (arr**lmbda - 1) / lmbda
        # Winsorize
        low, high = self.winsor_limits
        for col in self.winsor_cols:
            X[f"{col}_win"] = winsorize(X[col], limits=[low, high])
        return X

# 1. Load df_daily
df_daily = pd.read_parquet('df_daily.parquet')

# 2. Buat kolom return (simple percent change) dan drop NA awal
df_daily['return'] = df_daily['Close'].pct_change()
df_daily = df_daily.dropna(subset=['return']).reset_index(drop=True)

# 3. Terapkan DistributionTransformer
dist_tf = DistributionTransformer(
    yeojohnson_cols=['return'],
    boxcox_cols=['Volume'],
    winsor_cols=['Close', 'return'],
    winsor_limits=(0.01, 0.01)
)
df_transformed = dist_tf.fit_transform(df_daily)

# 4. Definisikan target klasifikasi & regresi
df = df_transformed.copy()
df['target_class'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df['target_reg']   = np.log(df['Close'].shift(-1)) - np.log(df['Close'])
df = df.dropna(subset=['target_class', 'target_reg']).reset_index(drop=True)

# 5. Cek distribusi kelas
print("Distribusi kelas:\n", df['target_class'].value_counts(normalize=True))

# 6. Siapkan X, y
X       = df.drop(columns=['target_class', 'target_reg'])
y_class = df['target_class']
y_reg   = df['target_reg']

# X, y_class, y_reg siap untuk split time-series & training model


> Siapkan Data Transformasi Akhir

In [ ]:
# Misal df sudah berisi df_transformed + kolom target_class & target_reg

# 1. Pilih hanya kolom fitur (semua selain target dan kolom mentah yang tidak perlu)
feature_cols = [
    # semua kolom transformasi distribusi
    col for col in df.columns 
    if col.endswith(('_yj', '_bc', '_win'))
] + [
    # fitur sentimen dan indikator teknikal & kalender lainnya
    'sentiment', 'Volume', 'Open', 'High', 'Low',
    # tambahkan kolom-kolom lag/rolling/kalender Anda di sini, misal:
    # 'return_yj_lag1', 'SMA7', 'RSI14', 'day_of_week', ...
]

# Pastikan feature_cols hanya memuat kolom yang ada
feature_cols = [c for c in feature_cols if c in df.columns]

# 2. Buat X_final dan y_final (pilih target sesuai problem)
X_final   = df[feature_cols].copy()
y_class   = df['target_class']
y_regress = df['target_reg']

print("Shape X_final:", X_final.shape)
print("Contoh fitur:\n", X_final.head())
print("Distribusi target_class:\n", y_class.value_counts(normalize=True))


> Train-Validation-Test Split Kronologis

In [ ]:
# Asumsikan X_final, y_class, y_regress sudah tersedia
import numpy as np

# 1. Tentukan proporsi split
n = len(X_final)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

# 2. Split kronologis
X_train, y_train_class, y_train_reg = (
    X_final.iloc[:train_end],
    y_class.iloc[:train_end],
    y_regress.iloc[:train_end]
)
X_val, y_val_class, y_val_reg = (
    X_final.iloc[train_end:val_end],
    y_class.iloc[train_end:val_end],
    y_regress.iloc[train_end:val_end]
)
X_test, y_test_class, y_test_reg = (
    X_final.iloc[val_end:],
    y_class.iloc[val_end:],
    y_regress.iloc[val_end:]
)

# 3. Cek ukuran masing‐masing set
print(f"Train    : {X_train.shape[0]} baris")
print(f"Validation: {X_val.shape[0]} baris")
print(f"Test     : {X_test.shape[0]} baris")

# 4. Verifikasi kontinuitas (opsional)
print("Index terakhir train:", X_train.index[-1])
print("Index pertama val :", X_val.index[0])
print("Index terakhir val :", X_val.index[-1])
print("Index pertama test:", X_test.index[0])


> TimeSeries Cross-Validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 1. Definisikan TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# 2. Cek indeks tiap fold
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"Fold {i+1}:")
    print("  Train:", train_idx[0], "to", train_idx[-1])
    print("  Val  :", val_idx[0], "to", val_idx[-1])

# 3. Buat pipeline contoh
pipe = Pipeline([
    ('scaler', StandardScaler()),          # scaling fitur numerik
    ('clf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'           # optional, tapi distribusi cukup seimbang
    ))
])

# 4. Cross‐val score untuk klasifikasi (misal AUC)
scores = cross_val_score(
    pipe,
    X_train, y_train_class,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1
)
print("CV AUC per fold:", scores)
print("Mean AUC       :", scores.mean())

# 5. Untuk regresi ganti model dan scoring:
from sklearn.ensemble import RandomForestRegressor
pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])
scores_reg = cross_val_score(
    pipe_reg,
    X_train, y_train_reg,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
print("CV MSE per fold:", -scores_reg)
print("Mean MSE       :", -scores_reg.mean())


> Pipeline Preprocessing

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer

# 1. Identifikasi tipe fitur
numeric_features = X_train.columns.tolist()  # semua kolom adalah numerik
# Jika ada fitur kategori, pisahkan ke categorical_features = [...]

# 2. Buat sub‐pipeline untuk fitur numerik
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),  # ganti strategy jika perlu
    ('scaler', StandardScaler()),                                   # atau QuantileTransformer(output_distribution='normal')
])

# 3. (Opsional) Sub‐pipeline untuk fitur kategori
# categorical_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('onehot', OneHotEncoder(handle_unknown='ignore'))
# ])

# 4. Gabungkan dengan ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    # ('cat', categorical_pipeline, categorical_features)
])

# 5. Contoh full pipeline untuk klasifikasi
from sklearn.ensemble import RandomForestClassifier

clf_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

# 6. Contoh full pipeline untuk regresi
from sklearn.ensemble import RandomForestRegressor

reg_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('reg', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# Sekarang 'clf_pipeline' dan 'reg_pipeline' siap untuk fit(), cross_val_score(), atau GridSearchCV().


>Inisialisasi Base Models

> Simple TabTransformer

In [ ]:
# 1. Install (sekali saja)
!pip install pytorch-tabular[all]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
import numpy as np

# 1. Definisi model
class SimpleTabTransformer(nn.Module):
    def __init__(self, input_dim, embed_dim=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        # Proyeksi fitur numerik ke embedding
        self.feature_embed = nn.Linear(input_dim, embed_dim)
        # Positional / feature token (satu per batch)
        self.pos_embed = nn.Parameter(torch.randn(1, 1, embed_dim))
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dropout=dropout,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        # Head klasifikasi
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2)
        )

    def forward(self, x):
        # x: [B, F]
        h = self.feature_embed(x)             # [B, E]
        h = h.unsqueeze(1) + self.pos_embed   # [B, 1, E]
        h = self.transformer(h)               # [B, 1, E]
        h = h.squeeze(1)                      # [B, E]
        return self.classifier(h)             # [B, 2]

# 2. Siapkan DataLoader (tanpa shuffle untuk time-series)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train_class.values, dtype=torch.long)
train_ds   = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)

X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val_class.values, dtype=torch.long)
val_ds   = TensorDataset(X_val_t, y_val_t)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

# 3. Inisialisasi model, loss, optimizer
model = SimpleTabTransformer(input_dim=X_train.shape[1], embed_dim=64, n_heads=4, n_layers=2, dropout=0.1)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. Training loop dengan validasi tiap epoch
best_auc = 0.0
for epoch in range(1, 11):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    
    # Validasi
    model.eval()
    all_probs, all_truth = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = nn.functional.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_probs.append(probs)
            all_truth.append(yb.numpy())
    all_probs = np.concatenate(all_probs)
    all_truth = np.concatenate(all_truth)
    auc = roc_auc_score(all_truth, all_probs)
    
    print(f"Epoch {epoch:02d} — Train Loss: {avg_loss:.4f}, Val AUC: {auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "best_tabtransformer.pth")

print(f"Best Validation AUC: {best_auc:.4f}")


> TabNet

In [ ]:
import warnings
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# 0. Sembunyikan peringatan spesifik dari pytorch-tabnet callbacks
warnings.filterwarnings("ignore", message="Device used :")
warnings.filterwarnings("ignore", message="Best weights from best epoch are automatically used!")

# 1. Siapkan data numpy
X_train_np = X_train.values
y_train_np = y_train_class.values
X_val_np   = X_val.values
y_val_np   = y_val_class.values

# 2. Pilih device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Inisialisasi TabNet tanpa verbose
tabnet = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 2e-3},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={'step_size': 10, 'gamma': 0.9},
    mask_type='entmax',
    verbose=0,
    device_name=device
)

# 4. Latih baseline (early stopping tetap aktif agar best weights terpakai,
#    tapi warning sudah disembunyikan)
tabnet.fit(
    X_train=X_train_np, y_train=y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=64
)

# 5. Evaluasi AUC pada validation set
probs = tabnet.predict_proba(X_val_np)[:, 1]
auc   = roc_auc_score(y_val_np, probs)
print(f"Validation AUC (TabNet Baseline): {auc:.4f}")

> CatBoost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# 1. Inisialisasi CatBoostClassifier baseline
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_seed=42,
    verbose=100,              # log setiap 100 iterasi
    early_stopping_rounds=50  # berhenti jika 50 iterasi tanpa peningkatan AUC
)

# 2. Latih model pada split kronologis
#    X_train, y_train_class, X_val, y_val_class telah disiapkan sebelumnya
cat_model.fit(
    X_train, 
    y_train_class,
    eval_set=(X_val, y_val_class),
    use_best_model=True
)

# 3. Prediksi probabilitas kelas “up” pada validation set
probs = cat_model.predict_proba(X_val)[:, 1]

# 4. Hitung AUC sebagai metrik evaluasi
auc = roc_auc_score(y_val_class, probs)
print(f"Validation AUC (CatBoost Baseline): {auc:.4f}")

# 5. Tampilkan 10 fitur terpenting
fi = pd.Series(cat_model.get_feature_importance(), index=X_train.columns)
print("Top 10 fitur penting:\n", fi.nlargest(10))


LightGBM

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score
import pandas as pd

# 1. Inisialisasi LGBMClassifier baseline
lgb_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42
)

# 2. Latih model dengan early stopping via callback
lgb_model.fit(
    X_train, 
    y_train_class,
    eval_set=[(X_val, y_val_class)],
    eval_metric='auc',
    callbacks=[
        early_stopping(stopping_rounds=50, verbose=False),
        log_evaluation(period=100)  # log setiap 100 iterasi
    ]
)

# 3. Prediksi probabilitas “up” pada validation set
probs = lgb_model.predict_proba(X_val)[:, 1]

# 4. Hitung AUC sebagai metrik evaluasi
auc = roc_auc_score(y_val_class, probs)
print(f"Validation AUC (LightGBM Baseline): {auc:.4f}")

# 5. Tampilkan top-10 fitur terpenting
fi = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
print("Top 10 fitur penting:\n", fi.nlargest(10))


>Training Default & Evaluasi CV

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
from torch.utils.data import DataLoader, TensorDataset

# --- 1. Siapkan splitter CV ---
tscv = TimeSeriesSplit(n_splits=5)

# --- 2. Persiapan container hasil ---
cv_results = {
    "TabTransformer": [],
    "TabNet"        : [],
    "CatBoost"      : [],
    "LightGBM"      : [],
}

# --- 3. Loop tiap fold ---
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    print(f"\n=== Fold {fold} ===")
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train_class.iloc[train_idx], y_train_class.iloc[val_idx]

    # -------- 3a. Simple TabTransformer --------
    # (gunakan SimpleTabTransformer dari kode Anda sebelumnya)
    model_tt = SimpleTabTransformer(input_dim=X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model_tt.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    # DataLoader tanpa shuffle
    tr_ds = TensorDataset(torch.tensor(X_tr.values, dtype=torch.float32),
                          torch.tensor(y_tr.values, dtype=torch.long))
    va_ds = TensorDataset(torch.tensor(X_va.values, dtype=torch.float32),
                          torch.tensor(y_va.values, dtype=torch.long))
    tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=False)
    va_loader = DataLoader(va_ds, batch_size=256, shuffle=False)
    # 3 epoch singkat untuk baseline
    for _ in range(3):
        model_tt.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model_tt(xb), yb)
            loss.backward()
            optimizer.step()
    # Prediksi dan AUC
    model_tt.eval()
    probs_tt, truths = [], []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb = xb.to(device)
            logits = model_tt(xb)
            probs_tt.append(torch.softmax(logits, 1)[:,1].cpu().numpy())
            truths.append(yb.numpy())
    probs_tt = np.concatenate(probs_tt)
    truths  = np.concatenate(truths)
    cv_results["TabTransformer"].append(roc_auc_score(truths, probs_tt))

    # -------- 3b. TabNet --------
    tabnet = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5, gamma=1.5, lambda_sparse=1e-3,
        optimizer_fn=torch.optim.Adam, optimizer_params={'lr':2e-3},
        mask_type='entmax', verbose=0, device_name=device
    )
    tabnet.fit(
        X_train=X_tr.values, y_train=y_tr.values,
        eval_set=[(X_va.values, y_va.values)],
        eval_metric=['auc'], max_epochs=20, patience=5,
        batch_size=256, virtual_batch_size=64
    )
    probs_tn = tabnet.predict_proba(X_va.values)[:,1]
    cv_results["TabNet"].append(roc_auc_score(y_va, probs_tn))

    # -------- 3c. CatBoost --------
    cb = CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=6,
        eval_metric='AUC', random_seed=42, verbose=0, early_stopping_rounds=30
    )
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    probs_cb = cb.predict_proba(X_va)[:,1]
    cv_results["CatBoost"].append(roc_auc_score(y_va, probs_cb))

    # -------- 3d. LightGBM --------
    lgb = LGBMClassifier(
        n_estimators=500, learning_rate=0.05,
        num_leaves=31, colsample_bytree=0.8, subsample=0.8, random_state=42
    )
    lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)], eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=30, verbose=False),
                   log_evaluation(period=0)]
    )
    probs_lgb = lgb.predict_proba(X_va)[:,1]
    cv_results["LightGBM"].append(roc_auc_score(y_va, probs_lgb))

# --- 4. Ringkasan hasil CV ---
summary = {
    model: (np.mean(scores), np.std(scores))
    for model, scores in cv_results.items()
}
df_summary = pd.DataFrame(summary, index=['mean_auc','std_auc']).T
print("\n=== CV Summary ===")
print(df_summary)


>Analisis Learning Curve

In [ ]:
!pip install catboost lightgbm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, TimeSeriesSplit
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

def plot_sklearn_learning_curve(model, name):
    tscv = TimeSeriesSplit(n_splits=5)
    train_sizes, train_scores, val_scores = learning_curve(
        estimator=model,
        X=X_train, y=y_train_class,
        cv=tscv,
        scoring='roc_auc',
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=1                 # gunakan single‐thread
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std  = np.std(train_scores, axis=1)
    val_mean   = np.mean(val_scores, axis=1)
    val_std    = np.std(val_scores, axis=1)

    plt.figure(figsize=(6,4))
    plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2)
    plt.fill_between(train_sizes, val_mean-val_std,   val_mean+val_std,   alpha=0.2)
    plt.plot(train_sizes, train_mean, 'o-', label=f'{name} Train AUC')
    plt.plot(train_sizes, val_mean,   'o-', label=f'{name} Val AUC')
    plt.xlabel('Proporsi Data Training')
    plt.ylabel('ROC AUC')
    plt.title(f'Learning Curve – {name}')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot untuk CatBoost
cb = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    eval_metric='AUC', verbose=0
)
plot_sklearn_learning_curve(cb, 'CatBoost')

# Plot untuk LightGBM
lgb = LGBMClassifier(
    n_estimators=500, learning_rate=0.05,
    num_leaves=31, colsample_bytree=0.8, subsample=0.8,
    random_state=42
)
plot_sklearn_learning_curve(lgb, 'LightGBM')


> Simple TabTransformer (manual)

In [ ]:
import os
# Pastikan PyTorch hanya menggunakan CPU
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

# 1. Set device ke CPU
device = torch.device('cpu')

fractions = np.linspace(0.2, 1.0, 5)
train_auc, val_auc = [], []

for frac in fractions:
    # 2. Subset kronologis
    n_sub = int(len(X_train) * frac)
    X_sub, y_sub = X_train.iloc[:n_sub], y_train_class.iloc[:n_sub]
    split = int(n_sub * 0.8)
    X_tr, X_va = X_sub.iloc[:split], X_sub.iloc[split:]
    y_tr, y_va = y_sub.iloc[:split], y_sub.iloc[split:]

    # 3. DataLoader (CPU)
    tr_ds = TensorDataset(
        torch.tensor(X_tr.values, dtype=torch.float32),
        torch.tensor(y_tr.values, dtype=torch.long)
    )
    va_ds = TensorDataset(
        torch.tensor(X_va.values, dtype=torch.float32),
        torch.tensor(y_va.values, dtype=torch.long)
    )
    tr_ld = DataLoader(tr_ds, batch_size=128, shuffle=False)
    va_ld = DataLoader(va_ds, batch_size=128, shuffle=False)

    # 4. Inisialisasi SimpleTabTransformer
    model = SimpleTabTransformer(input_dim=X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()

    # 5. Training singkat (3 epoch)
    for _ in range(3):
        model.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    # 6. Evaluasi AUC train
    model.eval()
    probs_tr, tr_truth = [], []
    with torch.no_grad():
        for xb, yb in tr_ld:
            xb = xb.to(device)
            p = torch.softmax(model(xb), dim=1)[:,1].cpu().numpy()
            probs_tr.append(p)
            tr_truth.append(yb.numpy())
    train_auc.append(roc_auc_score(
        np.concatenate(tr_truth),
        np.concatenate(probs_tr)
    ))

    # 7. Evaluasi AUC val
    probs_va, va_truth = [], []
    with torch.no_grad():
        for xb, yb in va_ld:
            xb = xb.to(device)
            p = torch.softmax(model(xb), dim=1)[:,1].cpu().numpy()
            probs_va.append(p)
            va_truth.append(yb.numpy())
    val_auc.append(roc_auc_score(
        np.concatenate(va_truth),
        np.concatenate(probs_va)
    ))

# 8. Plot learning curve
plt.figure(figsize=(6,4))
plt.plot(fractions, train_auc, 'o-', label='Train AUC')
plt.plot(fractions, val_auc,   'o-', label='Val AUC')
plt.xlabel('Fraksi Data Training')
plt.ylabel('ROC AUC')
plt.title('Learning Curve – Simple TabTransformer')
plt.legend()
plt.tight_layout()
plt.show()


> TabNet (manual)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import roc_auc_score

fractions = np.linspace(0.2, 1.0, 5)
train_auc, val_auc = [], []

for frac in fractions:
    n_sub = int(len(X_train) * frac)
    X_sub, y_sub = X_train.iloc[:n_sub].values, y_train_class.iloc[:n_sub].values
    split = int(n_sub * 0.8)
    X_tr, X_va = X_sub[:split], X_sub[split:]
    y_tr, y_va = y_sub[:split], y_sub[split:]

    model = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5, gamma=1.5, lambda_sparse=1e-3,
        optimizer_fn=torch.optim.Adam, optimizer_params={'lr':2e-3},
        mask_type='entmax', verbose=0, device_name=device
    )
    model.fit(
        X_train=X_tr, y_train=y_tr,
        eval_set=[(X_va, y_va)],
        max_epochs=20, patience=5,
        batch_size=256, virtual_batch_size=64
    )

    # AUC train
    probs_tr = model.predict_proba(X_tr)[:,1]
    train_auc.append(roc_auc_score(y_tr, probs_tr))
    # AUC val
    probs_va = model.predict_proba(X_va)[:,1]
    val_auc.append(roc_auc_score(y_va, probs_va))

# plot
plt.plot(fractions, train_auc, 'o-', label='Train AUC')
plt.plot(fractions, val_auc,   'o-', label='Val AUC')
plt.xlabel('Fraksi Data Training')
plt.ylabel('ROC AUC')
plt.title('Learning Curve – TabNet')
plt.legend()
plt.show()


>Hyperparameter Tuning Terfokus

>Simple TabTransformer (PyTorch + Optuna)

In [ ]:
import optuna
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

def objective_tt(trial):
    # 1) Sample hyperparams
    embed_dim  = trial.suggest_categorical("embed_dim", [32, 64, 128])
    n_heads    = trial.suggest_categorical("n_heads", [2, 4, 8])
    n_layers   = trial.suggest_int("n_layers", 1, 3)
    dropout    = trial.suggest_float("dropout", 0.0, 0.3)
    lr         = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    # 2) Prepare small CV fold
    tscv = TimeSeriesSplit(n_splits=3)
    aucs = []
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train_class.iloc[train_idx], y_train_class.iloc[val_idx]
        # DataLoader
        tr_ds = TensorDataset(torch.tensor(X_tr.values, dtype=torch.float32),
                              torch.tensor(y_tr.values, dtype=torch.long))
        va_ds = TensorDataset(torch.tensor(X_va.values, dtype=torch.float32),
                              torch.tensor(y_va.values, dtype=torch.long))
        tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=False)
        va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False)

        # Model init
        model = SimpleTabTransformer(
            input_dim=X_train.shape[1],
            embed_dim=embed_dim, n_heads=n_heads, n_layers=n_layers, dropout=dropout
        ).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        crit = torch.nn.CrossEntropyLoss()

        # Train 3 epochs
        for _ in range(3):
            model.train()
            for xb, yb in tr_ld:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = crit(model(xb), yb)
                loss.backward()
                opt.step()

        # Eval AUC
        model.eval()
        preds, truths = [], []
        with torch.no_grad():
            for xb, yb in va_ld:
                xb = xb.to(device)
                probs = torch.softmax(model(xb),1)[:,1].cpu().numpy()
                preds.append(probs); truths.append(yb.numpy())
        aucs.append(roc_auc_score(np.concatenate(truths), np.concatenate(preds)))

    return np.mean(aucs)

study_tt = optuna.create_study(direction="maximize")
study_tt.optimize(objective_tt, n_trials=20)
print("Best TabTransformer params:", study_tt.best_trial.params)


>TabNet (Optuna)

In [ ]:
import optuna
from pytorch_tabnet.tab_model import TabNetClassifier

def objective_tn(trial):
    n_d            = trial.suggest_categorical("n_d", [32, 64, 128])
    n_a            = trial.suggest_categorical("n_a", [32, 64, 128])
    n_steps        = trial.suggest_int("n_steps", 3, 7)
    gamma          = trial.suggest_float("gamma", 1.0, 2.0)
    lambda_sparse  = trial.suggest_loguniform("lambda_sparse", 1e-5, 1e-2)
    lr             = trial.suggest_loguniform("lr", 1e-4, 1e-2)

    tscv = TimeSeriesSplit(n_splits=3)
    aucs = []
    for tr_idx, va_idx in tscv.split(X_train):
        X_tr, y_tr = X_train.values[tr_idx], y_train_class.values[tr_idx]
        X_va, y_va = X_train.values[va_idx], y_train_class.values[va_idx]

        model = TabNetClassifier(
            n_d=n_d, n_a=n_a, n_steps=n_steps,
            gamma=gamma, lambda_sparse=lambda_sparse,
            optimizer_params={'lr': lr},
            verbose=0, device_name=device
        )
        model.fit(
            X_train=X_tr, y_train=y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric=['auc'], max_epochs=20,
            patience=5, batch_size=128, virtual_batch_size=64
        )
        preds = model.predict_proba(X_va)[:,1]
        aucs.append(roc_auc_score(y_va, preds))

    return np.mean(aucs)

study_tn = optuna.create_study(direction="maximize")
study_tn.optimize(objective_tn, n_trials=20)
print("Best TabNet params:", study_tn.best_trial.params)


> CatBoost (sklearn API + Optuna)

In [ ]:
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

def objective_cb(trial):
    params = {
        'iterations'          : trial.suggest_int("iterations", 200, 1000),
        'learning_rate'       : trial.suggest_loguniform("lr", 1e-3, 1e-1),
        'depth'               : trial.suggest_int("depth", 4, 10),
        'l2_leaf_reg'         : trial.suggest_loguniform("l2", 1e-3, 10),
        'random_seed'         : 42,
        'verbose'             : False,
        'early_stopping_rounds': 30
    }
    tscv = TimeSeriesSplit(n_splits=3)
    aucs = []
    for tr_idx, va_idx in tscv.split(X_train):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train_class.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx], y_train_class.iloc[va_idx]
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        preds = model.predict_proba(X_va)[:,1]
        aucs.append(roc_auc_score(y_va, preds))
    return np.mean(aucs)

study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(objective_cb, n_trials=20)
print("Best CatBoost params:", study_cb.best_trial.params)

> LightGBM (Optuna)

In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import TimeSeriesSplit

def objective_lgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int("n_estimators", 200, 1000),
        'learning_rate'   : trial.suggest_loguniform("lr", 1e-3, 1e-1),
        'num_leaves'      : trial.suggest_int("num_leaves", 16, 64),
        'colsample_bytree': trial.suggest_float("colsample", 0.6, 1.0),
        'subsample'       : trial.suggest_float("subsample", 0.6, 1.0),
        'random_state'    : 42
    }
    tscv = TimeSeriesSplit(n_splits=3)
    aucs = []
    for tr_idx, va_idx in tscv.split(X_train):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train_class.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx], y_train_class.iloc[va_idx]
        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric='auc',
            callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
        )
        preds = model.predict_proba(X_va)[:,1]
        aucs.append(roc_auc_score(y_va, preds))
    return np.mean(aucs)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=20)
print("Best LightGBM params:", study_lgb.best_trial.params)


>Evaluasi Hasil Tuning

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from pytorch_tabnet.tab_model import TabNetClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# --- 1. Gabungkan train + val ---
X_trval = pd.concat([X_train, X_val], axis=0)
y_trval = pd.concat([y_train_class, y_val_class], axis=0)

# --- 2. Simple TabTransformer (retrain) ---
# Buat DataLoader CPU-only
device = torch.device('cpu')
tr_ds = TensorDataset(
    torch.tensor(X_trval.values, dtype=torch.float32),
    torch.tensor(y_trval.values, dtype=torch.long)
)
tr_ld = DataLoader(tr_ds, batch_size=128, shuffle=False)

tt = SimpleTabTransformer(
    input_dim=X_trval.shape[1],
    embed_dim=32,
    n_heads=8,
    n_layers=1,
    dropout=0.24982529523101063
).to(device)
opt_tt = torch.optim.Adam(tt.parameters(), lr=0.0004436020733991433)
crit = torch.nn.CrossEntropyLoss()
# Latih 3 epoch sebagai baseline penuh
for _ in range(3):
    tt.train()
    for xb, yb in tr_ld:
        xb, yb = xb.to(device), yb.to(device)
        opt_tt.zero_grad()
        loss = crit(tt(xb), yb)
        loss.backward()
        opt_tt.step()

# --- 3. TabNet (retrain) ---
tn = TabNetClassifier(
    n_d=32, n_a=32, n_steps=6,
    gamma=1.875003791300602,
    lambda_sparse=0.003232203127399565,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr':0.009759611272018444},
    mask_type='entmax',
    verbose=0,
    device_name='cpu'
)
tn.fit(
    X_train=X_trval.values, y_train=y_trval.values,
    eval_set=[(X_val.values, y_val_class.values)],
    eval_metric=['auc'], max_epochs=50, patience=10,
    batch_size=256, virtual_batch_size=64
)

# --- 4. CatBoost (retrain) ---
cb_final = CatBoostClassifier(
    iterations=203,
    learning_rate=0.03590318015078008,
    depth=10,
    l2_leaf_reg=0.02821958370288102,
    eval_metric='AUC',
    random_seed=42,
    verbose=0,
    early_stopping_rounds=30
)
cb_final.fit(X_trval, y_trval, use_best_model=True)

# --- 5. LightGBM (retrain) ---
lgb_final = LGBMClassifier(
    n_estimators=932,
    learning_rate=0.009024801542521551,
    num_leaves=22,
    colsample_bytree=0.617898589070388,
    subsample=0.662121769863855,
    random_state=42
)
lgb_final.fit(
    X_trval, y_trval,
    eval_set=[(X_val, y_val_class)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=30, verbose=False),
               log_evaluation(period=0)]
)

# --- 6. Evaluasi pada test set ---
# Probabilitas “up” (kelas 1) untuk TabTransformer, dengan detach()
with torch.no_grad():
    logits_tt = tt(torch.tensor(X_test.values, dtype=torch.float32).to(device))
    probs_tt  = torch.softmax(logits_tt, dim=1)[:, 1].detach().cpu().numpy()

probs_tn  = tn.predict_proba(X_test.values)[:,1]
probs_cb  = cb_final.predict_proba(X_test)[:,1]
probs_lgb = lgb_final.predict_proba(X_test)[:,1]

# Hitung metrik
probs_dict = {
    'TabTransformer': probs_tt,
    'TabNet'        : probs_tn,
    'CatBoost'      : probs_cb,
    'LightGBM'      : probs_lgb
}
results = {}
for name, probs in probs_dict.items():
    preds = (probs > 0.5).astype(int)
    results[name] = {
        'AUC'      : roc_auc_score(y_test_class, probs),
        'Accuracy' : accuracy_score(y_test_class, preds),
        'Precision': precision_score(y_test_class, preds),
        'Recall'   : recall_score(y_test_class, preds),
        'F1'       : f1_score(y_test_class, preds)
    }

# Ringkas & tampilkan
df_results = pd.DataFrame(results).T
print(df_results)

>Analisis Feature Importance & Interpretasi

>CatBoost — Built-in Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1.1 Ekstrak importance
fi_cb = pd.Series(
    cb_final.get_feature_importance(type='FeatureImportance'),
    index=X_trval.columns
).sort_values(ascending=False)

# 1.2 Tampilkan top-10
print("CatBoost Top-10 Feature Importance:")
print(fi_cb.head(10).to_frame(name='Importance'))

# 1.3 Plot bar chart
plt.figure(figsize=(6,4))
fi_cb.head(10).plot(kind='bar')
plt.title('CatBoost Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

>LightGBM — Built-in Importance

In [ ]:
# 2.1 Ekstrak importance
fi_lgb = pd.Series(
    lgb_final.feature_importances_,
    index=X_trval.columns
).sort_values(ascending=False)

# 2.2 Tampilkan top-10
print("LightGBM Top-10 Feature Importance:")
print(fi_lgb.head(10).to_frame(name='Importance'))

# 2.3 Plot
plt.figure(figsize=(6,4))
fi_lgb.head(10).plot(kind='bar')
plt.title('LightGBM Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

> TabNet — Sparse Attention Masks

In [ ]:
# 3.1 Ekstrak feature importances dari TabNet
fi_tn = pd.Series(
    tn.feature_importances_,
    index=X_trval.columns
).sort_values(ascending=False)

# 3.2 Tampilkan top-10
print("TabNet Top-10 Feature Importance:")
print(fi_tn.head(10).to_frame(name='Importance'))

# 3.3 Plot
plt.figure(figsize=(6,4))
fi_tn.head(10).plot(kind='bar')
plt.title('TabNet Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

> Simple TabTransformer — SHAP Values

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm  # untuk progress bar

# 1. Hitung AUC baseline
with torch.no_grad():
    logits = tt(torch.tensor(X_test.values, dtype=torch.float32).to(device))
    probs_tt = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
baseline_auc = roc_auc_score(y_test_class, probs_tt)

# 2. Fungsi bantu mem-prediksi dengan TT
def tt_auc_on_df(df):
    with torch.no_grad():
        logits = tt(torch.tensor(df.values, dtype=torch.float32).to(device))
        probs  = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
    return roc_auc_score(y_test_class, probs)

# 3. Loop drop‐column
importances = {}
for col in tqdm(X_test.columns, desc="Drop‐column importance"):
    df_perm = X_test.copy()
    df_perm[col] = np.random.permutation(df_perm[col].values)
    perm_auc = tt_auc_on_df(df_perm)
    importances[col] = baseline_auc - perm_auc

# 4. Ringkas top‐10
imp_series = pd.Series(importances).sort_values(ascending=False)
top10 = imp_series.head(10)
print("SimpleTabTransformer Drop‐column Importance (Top 10):")
print(top10.to_frame(name="ΔAUC"))

# 5. Plot bar chart
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
top10.plot(kind='bar')
plt.ylabel('Decrease in AUC when shuffled')
plt.title('Feature Importance – Simple TabTransformer')
plt.tight_layout()
plt.show()


>Final Testing

> Ringkasan Metrik pada Test Set

In [ ]:
import pandas as pd

# Asumsikan df_results sudah berisi metrik AUC, Accuracy, Precision, Recall, F1
# Jika belum, jalankan kembali block evaluasi berikut:
df_results = pd.DataFrame({
    'TabTransformer': results['TabTransformer'],
    'TabNet'        : results['TabNet'],
    'CatBoost'      : results['CatBoost'],
    'LightGBM'      : results['LightGBM']
}).T

print("=== Test Set Performance ===")
print(df_results)


> ROC Curve Comparison

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(6,4))
for name, probs in probs_dict.items():
    fpr, tpr, _ = roc_curve(y_test_class, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test_class, probs):.3f})")

plt.plot([0,1],[0,1],'k--', linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


> Classification Report untuk Model Terbaik

In [ ]:
from sklearn.metrics import classification_report

# 1. Dapatkan nama model dengan AUC tertinggi
best_model_name = df_results['AUC'].idxmax()
print(f"Model terbaik berdasarkan AUC: {best_model_name}")

# 2. Ambil probabilitas dan prediksi biner dari probs_dict
best_probs = probs_dict[best_model_name]
best_preds = (best_probs > 0.5).astype(int)

# 3. Cetak classification report secara otomatis
print(f"\n=== Classification Report: {best_model_name} on Test Set ===")
print(classification_report(
    y_test_class,
    best_preds,
    target_names=['Down (0)', 'Up (1)'],
    digits=4
))

>Calibration & Threshold (klasifikasi)

> Probabilitas Calibration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve, classification_report

# 1. Hitung probabilitas un‐calibrated pada validation set
prob_val_uncal = cb_final.predict_proba(X_val)[:,1]

# 2. Fit logistic regression sederhana di atas probabilitas un‐calibrated → Platt Scaling
lr_cal = LogisticRegression(solver='lbfgs')
lr_cal.fit(
    prob_val_uncal.reshape(-1,1),
    y_val_class
)

# 3. Terapkan ke test set
prob_test_uncal = cb_final.predict_proba(X_test)[:,1]
prob_test_cal   = lr_cal.predict_proba(prob_test_uncal.reshape(-1,1))[:,1]

# 4. Buat calibration curve manual
def manual_calibration_curve(y_true, prob_pred, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    bin_ids = np.digitize(prob_pred, bins) - 1
    frac_pos = []
    mean_pred = []
    for i in range(n_bins):
        mask = bin_ids == i
        if np.sum(mask) > 0:
            frac_pos.append(np.mean(y_true[mask]))
            mean_pred.append(np.mean(prob_pred[mask]))
        else:
            frac_pos.append(np.nan)
            mean_pred.append(np.mean([(bins[i]+bins[i+1])/2]))
    return np.array(mean_pred), np.array(frac_pos)

mean_uncal, frac_uncal = manual_calibration_curve(y_test_class, prob_test_uncal)
mean_cal,   frac_cal   = manual_calibration_curve(y_test_class, prob_test_cal)

# 5. Plot calibration curve
plt.figure(figsize=(6,4))
plt.plot(mean_uncal, frac_uncal, 's-', label='Uncalibrated')
plt.plot(mean_cal,   frac_cal,   'o-', label='Calibrated (Platt)')
plt.plot([0,1],[0,1], 'k--', label='Perfectly calibrated')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve — CatBoost (Platt Scaling)')
plt.legend()
plt.tight_layout()
plt.show()

# 6. Cari threshold optimal di validation (F1) dan Youden’s J
prec, rec, thr_pr = precision_recall_curve(y_val_class, lr_cal.predict_proba(prob_val_uncal.reshape(-1,1))[:,1])
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-12)
best_idx   = np.argmax(f1_scores)
best_thr_f1 = thr_pr[best_idx]

fpr, tpr, thr_roc = roc_curve(y_val_class, lr_cal.predict_proba(prob_val_uncal.reshape(-1,1))[:,1])
youden     = tpr - fpr
best_idx2  = np.argmax(youden)
best_thr_roc = thr_roc[best_idx2]

print(f"Optimal threshold max F1: {best_thr_f1:.3f}, F1: {f1_scores[best_idx]:.3f}")
print(f"Optimal threshold Youden's J: {best_thr_roc:.3f}, J-stat: {youden[best_idx2]:.3f}")

# 7. Classification report di test set, pakai calibrated probs & best_thr_f1
preds_test = (prob_test_cal >= best_thr_f1).astype(int)
print("\nClassification Report (CatBoost, calibrated & thr=max F1):")
print(classification_report(
    y_test_class, preds_test,
    target_names=['Down (0)', 'Up (1)'],
    digits=4
))

> Threshold Optimization

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_curve

# 2.1 Dapatkan probabilitas ter‐kalibrasi pada validation set via lr_cal
prob_val_uncal = cb_final.predict_proba(X_val)[:,1]
probs_val = lr_cal.predict_proba(prob_val_uncal.reshape(-1,1))[:,1]

# 2.2 Cari threshold terbaik berdasarkan F1
prec, rec, thr_pr = precision_recall_curve(y_val_class, probs_val)
# hilangkan titik terakhir yang tidak punya threshold
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_thr_f1 = thr_pr[best_idx]
print(f"Optimal threshold (max F1): {best_thr_f1:.3f}, F1: {f1_scores[best_idx]:.3f}")

# 2.3 Cari threshold yang memaksimalkan Youden’s J dari ROC
fpr, tpr, thr_roc = roc_curve(y_val_class, probs_val)
youden = tpr - fpr
best_idx2 = np.argmax(youden)
best_thr_roc = thr_roc[best_idx2]
print(f"Optimal threshold (Youden's J): {best_thr_roc:.3f}, J-stat: {youden[best_idx2]:.3f}")


> Penggunaan Threshold di Test Set

In [ ]:
from sklearn.metrics import classification_report

# 3.1 Probabilitas ter‐kalibrasi test menggunakan lr_cal
prob_test_uncal = cb_final.predict_proba(X_test)[:,1]
probs_test_cal  = lr_cal.predict_proba(prob_test_uncal.reshape(-1,1))[:,1]

# 3.2 Prediksi dengan threshold F1 yang sudah dihitung sebelumnya
preds_test = (probs_test_cal >= best_thr_f1).astype(int)

# 3.3 Evaluasi kembali dengan classification_report
print("=== Classification Report after Calibration & Thresholding ===")
print(classification_report(
    y_test_class, 
    preds_test,
    target_names=['Down (0)', 'Up (1)'],
    digits=4
))

>Ensembling 

> Stacking Ensemble dengan Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# --- 1. Hitung probabilitas “up” pada validation set untuk setiap model ---
# 1.1 Simple TabTransformer
with torch.no_grad():
    logits_val_tt = tt(torch.tensor(X_val.values, dtype=torch.float32).to(device))
    probs_val_tt  = torch.softmax(logits_val_tt, dim=1)[:,1].cpu().numpy()

# 1.2 TabNet
probs_val_tn  = tn.predict_proba(X_val.values)[:,1]

# 1.3 CatBoost
probs_val_cb  = cb_final.predict_proba(X_val)[:,1]

# 1.4 LightGBM
probs_val_lgb = lgb_final.predict_proba(X_val)[:,1]

# --- 2. Susun meta‐feature untuk training meta‐model ---
meta_train = np.vstack([
    probs_val_tt,
    probs_val_tn,
    probs_val_cb,
    probs_val_lgb
]).T  # shape = (n_val_samples, 4)

# --- 3. Latih meta‐model Logistic Regression ---
meta_clf = LogisticRegression(solver='lbfgs')
meta_clf.fit(meta_train, y_val_class)

# --- 4. Hitung probabilitas pada test set untuk setiap model ---
with torch.no_grad():
    logits_tt_test = tt(torch.tensor(X_test.values, dtype=torch.float32).to(device))
    probs_tt_test  = torch.softmax(logits_tt_test, dim=1)[:,1].cpu().numpy()

probs_tn_test  = tn.predict_proba(X_test.values)[:,1]
probs_cb_test  = cb_final.predict_proba(X_test)[:,1]
probs_lgb_test = lgb_final.predict_proba(X_test)[:,1]

# --- 5. Susun meta‐feature untuk test set ---
meta_test = np.vstack([
    probs_tt_test,
    probs_tn_test,
    probs_cb_test,
    probs_lgb_test
]).T  # shape = (n_test_samples, 4)

# --- 6. Prediksi ensemble dan evaluasi ---
probs_ens = meta_clf.predict_proba(meta_test)[:,1]
preds_ens = (probs_ens >= 0.5).astype(int)

print("Stacking Ensemble — ROC AUC:", roc_auc_score(y_test_class, probs_ens))
print("\nClassification Report (Stacking Ensemble on Test Set):")
print(classification_report(
    y_test_class,
    preds_ens,
    target_names=['Down (0)', 'Up (1)'],
    digits=4
))

>Simpan Pipeline & Model

In [ ]:
!pip install catboost lightgbm pytorch-tabnet torch

In [ ]:
import os
import joblib
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

# 0. Utility untuk safe‐load
def safe_load(path, loader_fn):
    if os.path.exists(path):
        return loader_fn(path)
    else:
        print(f"[Warning] Tidak ada file: {path}; melewatkan.")
        return None

# 1. Load preprocessing (jika ada)
dist_tf = safe_load(
    "saved_models/distribution_transformer.pkl",
    lambda p: joblib.load(p)
)
# Jika tidak ada, pakai identity:
if dist_tf is None:
    class IdentityTransformer:
        def transform(self, X): return X
    dist_tf = IdentityTransformer()

# 2. Load feature names (jika ada)
feature_names = safe_load(
    "saved_models/feature_names.pkl",
    lambda p: joblib.load(p)
)
# Jika tidak ada, nanti pakai semua kolom df_new:
if feature_names is None:
    feature_names = None  # akan di-handle di fungsi predict

# 3. Load base‐models

# 3a. Simple TabTransformer
tt = None
conf_path = "saved_models/simple_tabtransformer_config.pkl"
state_path = "saved_models/simple_tabtransformer.pth"
if os.path.exists(conf_path) and os.path.exists(state_path):
    from your_transformer_module import SimpleTabTransformer
    config_tt = joblib.load(conf_path)
    tt = SimpleTabTransformer(**config_tt)
    tt.load_state_dict(torch.load(state_path, map_location="cpu"))
    tt.eval()
else:
    print("[Warning] SimpleTabTransformer model files tidak lengkap; melewatkan.")

# 3b. TabNet
tn = safe_load("saved_models/tabnet_model.pkl", lambda p: joblib.load(p))

# 3c. CatBoost
from catboost import CatBoostClassifier
cb_final = CatBoostClassifier()
if os.path.exists("saved_models/catboost_model.cbm"):
    cb_final.load_model("saved_models/catboost_model.cbm")
else:
    print("[Warning] File catboost_model.cbm tidak ditemukan; melewatkan.")

# 3d. LightGBM
import lightgbm as lgb
lgb_final = None
if os.path.exists("saved_models/lightgbm_model.txt"):
    lgb_final = lgb.Booster(model_file="saved_models/lightgbm_model.txt")
else:
    print("[Warning] File lightgbm_model.txt tidak ditemukan; melewatkan.")

# 4. Load meta‐model stacking
meta_clf = safe_load("saved_models/stacking_meta_clf.pkl", lambda p: joblib.load(p))
if meta_clf is None:
    print("[Warning] Meta-model stacking tidak ditemukan; pipeline inference tidak lengkap.")

# 5. Inference function
def predict_direction(df_new: pd.DataFrame):
    # a) select feature columns
    if feature_names:
        X = df_new[feature_names].copy()
    else:
        X = df_new.copy()
    # b) preprocessing
    X_proc = dist_tf.transform(X)
    X_np   = X_proc.values

    # c) base‐model probabilities
    probs = {}
    # TabTransformer
    if tt:
        with torch.no_grad():
            logits = tt(torch.tensor(X_np, dtype=torch.float32))
            probs['tt'] = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
    # TabNet
    if tn:
        probs['tn'] = tn.predict_proba(X_np)[:,1]
    # CatBoost
    probs['cb'] = cb_final.predict_proba(X_np)[:,1]
    # LightGBM
    if lgb_final:
        probs['lgb'] = lgb_final.predict(X_np)

    # d) stacking meta‐features
    if meta_clf and probs:
        meta_feats = np.vstack([probs[k] for k in ['tt','tn','cb','lgb'] if k in probs]).T
        ens_probs = meta_clf.predict_proba(meta_feats)[:,1]
        ens_preds = (ens_probs >= 0.5).astype(int)
        return ens_preds, ens_probs
    else:
        raise RuntimeError("Meta-model atau base-models tidak lengkap; tidak dapat melakukan ensemble.")

# Contoh pemakaian
# new_data = pd.read_parquet("new_bitcoin_data.parquet")
# preds, probs = predict_direction(new_data)


# Out-of-Fold Prediction

> Siapkan CV splitter

In [ ]:
import pandas as pd
import numpy as np
from copy import deepcopy
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# 0. DEFINISIKAN HYPERPARAMETERS ANDA
best_tt_params = {
    'input_dim': X_trval.shape[1],
    'embed_dim': 64,
    'n_heads':   4,
    'n_layers':  2,
    'dropout':   0.1,
    'lr':        1e-3,
    'epochs':    10,
    'batch_size':128
}
best_tn_params = {
    'n_d':        8,
    'n_a':        8,
    'lr':         1e-3,
    'max_epochs': 50,
}
best_cb_params = {
    'iterations':    200,
    'learning_rate': 0.05,
    'depth':         6,
}
best_lgb_params = {
    'n_estimators':   200,
    'learning_rate':  0.05,
    'max_depth':      7,
}
# Pastikan:
# from your_module import SimpleTabTransformer
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Setup CV splitter
tscv = TimeSeriesSplit(n_splits=5, gap=0)

# 2. Init OOF containers
oof_tt  = np.zeros(len(X_trval))
oof_tn  = np.zeros(len(X_trval))
oof_cb  = np.zeros(len(X_trval))
oof_lgb = np.zeros(len(X_trval))

# 3. Loop per-fold
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_trval), start=1):
    X_tr, X_val = X_trval.iloc[train_idx], X_trval.iloc[val_idx]
    y_tr, y_val = y_trval.iloc[train_idx], y_trval.iloc[val_idx]
    print(f"[Fold {fold}] Train={X_tr.shape}, Val={X_val.shape}")

    # a) Distribusi transform
    dist_fold = deepcopy(dist_tf)
    if hasattr(dist_fold, "fit"):
        dist_fold.fit(X_tr.values)
    X_tr_dist  = dist_fold.transform(X_tr.values)
    X_val_dist = dist_fold.transform(X_val.values)

    # b) Scaling/encoding
    pre_fold = deepcopy(preprocessor).fit(pd.DataFrame(X_tr_dist, columns=X_tr.columns))
    X_tr_prep  = pre_fold.transform(pd.DataFrame(X_tr_dist, columns=X_tr.columns))
    X_val_prep = pre_fold.transform(pd.DataFrame(X_val_dist, columns=X_val.columns))
    print(f"    After prep: Tr={X_tr_prep.shape}, Val={X_val_prep.shape}")

    # 1) SimpleTabTransformer
    tt_fold = SimpleTabTransformer(
        input_dim=best_tt_params['input_dim'],
        embed_dim=best_tt_params['embed_dim'],
        n_heads=best_tt_params['n_heads'],
        n_layers=best_tt_params['n_layers'],
        dropout=best_tt_params['dropout']
    ).to(device)
    optimizer = torch.optim.Adam(tt_fold.parameters(), lr=best_tt_params['lr'])
    criterion = nn.CrossEntropyLoss()
    ds = TensorDataset(
        torch.tensor(X_tr_prep, dtype=torch.float32),
        torch.tensor(y_tr.values, dtype=torch.long)
    )
    loader = DataLoader(ds, batch_size=best_tt_params['batch_size'], shuffle=False)
    tt_fold.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(tt_fold(xb), yb)
        loss.backward()
        optimizer.step()
    tt_fold.eval()
    with torch.no_grad():
        logits = tt_fold(torch.tensor(X_val_prep, dtype=torch.float32).to(device))
        probs_tt = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

    # 2) TabNet (revisi)
    # pisahkan arsitektur vs optimizer params
    tn_arch = {k:v for k,v in best_tn_params.items() if k not in ['lr','max_epochs']}
    tn_fold = TabNetClassifier(
        **tn_arch,
        optimizer_params={'lr': best_tn_params['lr']}
    )
    tn_fold.fit(
        X_tr_prep, y_tr,
        max_epochs=best_tn_params['max_epochs']
    )
    probs_tn = tn_fold.predict_proba(X_val_prep)[:,1]

    # 3) CatBoost
    cb_fold = CatBoostClassifier(**best_cb_params, verbose=0)
    cb_fold.fit(X_tr_prep, y_tr)
    probs_cb = cb_fold.predict_proba(X_val_prep)[:,1]

    # 4) LightGBM
    lgb_fold = LGBMClassifier(**best_lgb_params)
    lgb_fold.fit(X_tr_prep, y_tr)
    probs_lgb = lgb_fold.predict_proba(X_val_prep)[:,1]

    print(f"    Shapes preds → tt:{probs_tt.shape}, tn:{probs_tn.shape}, cb:{probs_cb.shape}, lgb:{probs_lgb.shape}")

    # Simpan OOF
    oof_tt[val_idx]  = probs_tt
    oof_tn[val_idx]  = probs_tn
    oof_cb[val_idx]  = probs_cb
    oof_lgb[val_idx] = probs_lgb

    print(f"    [Fold {fold}] AUC_tt={roc_auc_score(y_val, probs_tt):.3f}, AUC_cb={roc_auc_score(y_val, probs_cb):.3f}")

# 4. Build meta-features & Fit meta-model
df_oof = pd.DataFrame({
    'tt_oof':  oof_tt,
    'tn_oof':  oof_tn,
    'cb_oof':  oof_cb,
    'lgb_oof': oof_lgb,
}, index=X_trval.index)

meta_clf = LogisticRegression()
meta_clf.fit(df_oof, y_trval)
print("OOF meta-model AUC overall:", roc_auc_score(y_trval, meta_clf.predict_proba(df_oof)[:,1]))


> Inisialisasi container OOF

In [ ]:
import numpy as np

# Misal X_trval sudah berisi fitur gabungan train+val
n_samples = len(X_trval)

# Inisialisasi container OOF untuk tiap base-model
oof_tt  = np.zeros(n_samples)   # untuk SimpleTabTransformer
oof_tn  = np.zeros(n_samples)   # untuk TabNet
oof_cb  = np.zeros(n_samples)   # untuk CatBoost
oof_lgb = np.zeros(n_samples)   # untuk LightGBM

# Cek bahwa semua array sudah berukuran sama
print(oof_tt.shape, oof_tn.shape, oof_cb.shape, oof_lgb.shape)
# → (n_samples,) semua


> Loop per fold

In [ ]:
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_trval), start=1):
    # 1) Split data kronologis
    X_tr, X_val = X_trval.iloc[train_idx], X_trval.iloc[val_idx]
    y_tr, y_val = y_trval.iloc[train_idx], y_trval.iloc[val_idx]
    print(f"[Fold {fold}] Train={X_tr.shape}, Val={X_val.shape}")

    # 2) Distribusi transform per‐fold
    dist_fold = deepcopy(dist_tf)
    if hasattr(dist_fold, "fit"):
        dist_fold.fit(X_tr.values)
    X_tr_dist  = dist_fold.transform(X_tr.values)
    X_val_dist = dist_fold.transform(X_val.values)

    # 3) Scaling & encoding per‐fold
    pre_fold = deepcopy(preprocessor).fit(
        pd.DataFrame(X_tr_dist, columns=X_tr.columns)
    )
    X_tr_prep  = pre_fold.transform(pd.DataFrame(X_tr_dist, columns=X_tr.columns))
    X_val_prep = pre_fold.transform(pd.DataFrame(X_val_dist, columns=X_val.columns))

    # 4) Base‐model training & OOF prediction

    # a) SimpleTabTransformer
    tt = SimpleTabTransformer(
        input_dim=best_tt_params['input_dim'],
        embed_dim=best_tt_params['embed_dim'],
        n_heads=best_tt_params['n_heads'],
        n_layers=best_tt_params['n_layers'],
        dropout=best_tt_params['dropout']
    ).to(device)
    opt = torch.optim.Adam(tt.parameters(), lr=best_tt_params['lr'])
    loss_fn = nn.CrossEntropyLoss()
    ds_tr = TensorDataset(
        torch.tensor(X_tr_prep, dtype=torch.float32),
        torch.tensor(y_tr.values, dtype=torch.long)
    )
    loader = DataLoader(ds_tr, batch_size=best_tt_params['batch_size'], shuffle=False)
    tt.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(tt(xb), yb).backward()
        opt.step()
    tt.eval()
    with torch.no_grad():
        logits = tt(torch.tensor(X_val_prep, dtype=torch.float32).to(device))
        probs_tt = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

    # b) TabNet
    tn = TabNetClassifier(
        n_d=best_tn_params['n_d'], n_a=best_tn_params['n_a'],
        optimizer_params={'lr': best_tn_params['lr']}
    )
    tn.fit(X_tr_prep, y_tr, max_epochs=best_tn_params['max_epochs'])
    probs_tn = tn.predict_proba(X_val_prep)[:,1]

    # c) CatBoost
    cb = CatBoostClassifier(**best_cb_params, verbose=0)
    cb.fit(X_tr_prep, y_tr)
    probs_cb = cb.predict_proba(X_val_prep)[:,1]

    # d) LightGBM
    lgb = LGBMClassifier(**best_lgb_params)
    lgb.fit(X_tr_prep, y_tr)
    probs_lgb = lgb.predict_proba(X_val_prep)[:,1]

    # 5) Simpan OOF ke container
    oof_tt[val_idx]  = probs_tt
    oof_tn[val_idx]  = probs_tn
    oof_cb[val_idx]  = probs_cb
    oof_lgb[val_idx] = probs_lgb

    # 6) evaluasi per‐fold
    print(f"  AUC -- TT: {roc_auc_score(y_val, probs_tt):.3f}, "
          f"CB: {roc_auc_score(y_val, probs_cb):.3f}")

# setelah loop, oof_tt, oof_tn, oof_cb, oof_lgb siap dipakai untuk stacking  

> Validasi OOF

> Cek kelengkapan OOF

In [ ]:
import numpy as np

for name, arr in [('TT', oof_tt), ('TN', oof_tn),
                  ('CB', oof_cb), ('LGB', oof_lgb)]:
    assert arr.shape[0] == len(y_trval), f"{name} length mismatch"
    assert not np.isnan(arr).any(),      f"{name} contains NaN"
print("✅ Semua OOF arrays terisi penuh dan tanpa NaN.")

> Hitung AUC per base-model (OOF)

In [ ]:
from sklearn.metrics import roc_auc_score

auc_tt  = roc_auc_score(y_trval, oof_tt)
auc_tn  = roc_auc_score(y_trval, oof_tn)
auc_cb  = roc_auc_score(y_trval, oof_cb)
auc_lgb = roc_auc_score(y_trval, oof_lgb)

print(f"OOF AUC -- TT: {auc_tt:.4f}, TN: {auc_tn:.4f}, "
      f"CB: {auc_cb:.4f}, LGB: {auc_lgb:.4f}")

> Bangun DataFrame meta-features & evaluasi meta-model

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

df_oof = pd.DataFrame({
    'tt_oof':  oof_tt,
    'tn_oof':  oof_tn,
    'cb_oof':  oof_cb,
    'lgb_oof': oof_lgb,
}, index=X_trval.index)

# Latih ulang meta-model jika belum:
meta_clf = LogisticRegression()
meta_clf.fit(df_oof, y_trval)

# AUC meta-model
oof_meta_proba = meta_clf.predict_proba(df_oof)[:,1]
auc_meta = roc_auc_score(y_trval, oof_meta_proba)
print(f"OOF AUC Meta-model: {auc_meta:.4f}")


> Analisis korelasi antar meta-features

In [ ]:
corr = df_oof.corr()
print("Korelasi antar OOF-features:\n", corr)

> Visualisasi ROC curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

plt.figure(figsize=(8,6))
for name, arr in [('TT', oof_tt), ('TN', oof_tn),
                  ('CB', oof_cb), ('LGB', oof_lgb),
                  ('Meta', oof_meta_proba)]:
    fpr, tpr, _ = roc_curve(y_trval, arr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_trval, arr):.3f})")
plt.plot([0,1],[0,1],'k--',linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve OOF Predictions")
plt.legend(loc="lower right")
plt.show()

> Bangun meta-features DataFrame

In [ ]:
import pandas as pd

# Anda sudah memiliki keempat array OOF:
# oof_tt, oof_tn, oof_cb, oof_lgb
# Dan X_trval.index sebagai index kronologis train+val

df_oof = pd.DataFrame({
    'tt_oof':  oof_tt,   # probabilitas OOF dari TabTransformer
    'tn_oof':  oof_tn,   # probabilitas OOF dari TabNet
    'cb_oof':  oof_cb,   # probabilitas OOF dari CatBoost
    'lgb_oof': oof_lgb,  # probabilitas OOF dari LightGBM
}, index=X_trval.index)

# Cek sekilas hasilnya
print(df_oof.head())
print("\nShape:", df_oof.shape)

> Latih meta-model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Inisialisasi meta-model
meta_clf = LogisticRegression(max_iter=1000)

# 2. Fit pada meta-features OOF
meta_clf.fit(df_oof, y_trval)

# 3. Prediksi probabilitas & kelas pada data OOF
proba_meta = meta_clf.predict_proba(df_oof)[:, 1]
preds_meta = (proba_meta >= 0.5).astype(int)  # threshold = 0.5, ganti jika Anda punya threshold optimal

# 4. Evaluasi
auc   = roc_auc_score(y_trval, proba_meta)
acc   = accuracy_score(y_trval, preds_meta)
prec  = precision_score(y_trval, preds_meta)
rec   = recall_score(y_trval, preds_meta)
f1    = f1_score(y_trval, preds_meta)

print(f"Meta-model performance on OOF:")
print(f"  AUC      : {auc:.4f}")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-score : {f1:.4f}")

> Refit base‐models ke full train+val

In [ ]:
import os
import pandas as pd
import joblib

# Buat direktori untuk menyimpan jika belum ada
os.makedirs('saved_models', exist_ok=True)

# 1) Re-fit distribution transformer & preprocessor
dist_full = deepcopy(dist_tf)
if hasattr(dist_full, "fit"):
    dist_full.fit(X_trval.values)
X_trval_dist = dist_full.transform(X_trval.values)

preproc_full = deepcopy(preprocessor).fit(
    pd.DataFrame(X_trval_dist, columns=X_trval.columns)
)
X_trval_prep = preproc_full.transform(
    pd.DataFrame(X_trval_dist, columns=X_trval.columns)
)

# Simpan transformer
joblib.dump(dist_full,    'saved_models/dist_transformer_full.pkl')
joblib.dump(preproc_full, 'saved_models/preprocessor_full.pkl')


# 2) Re-fit SimpleTabTransformer
# Pastikan Anda telah: 
#   from your_module import SimpleTabTransformer
#   device = torch.device(...)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

tt_full = SimpleTabTransformer(
    input_dim=best_tt_params['input_dim'],
    embed_dim=best_tt_params['embed_dim'],
    n_heads=best_tt_params['n_heads'],
    n_layers=best_tt_params['n_layers'],
    dropout=best_tt_params['dropout']
).to(device)

optimizer = torch.optim.Adam(tt_full.parameters(), lr=best_tt_params['lr'])
criterion = nn.CrossEntropyLoss()
ds_full    = TensorDataset(
    torch.tensor(X_trval_prep, dtype=torch.float32),
    torch.tensor(y_trval.values, dtype=torch.long)
)
loader_full = DataLoader(ds_full, batch_size=best_tt_params['batch_size'], shuffle=False)

tt_full.train()
for epoch in range(best_tt_params['epochs']):
    for xb, yb in loader_full:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(tt_full(xb), yb)
        loss.backward()
        optimizer.step()

# Simpan state‐dict
torch.save(tt_full.state_dict(), 'saved_models/tabtransformer_full.pth')


# 3) Re-fit TabNetClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

tn_arch = {k: v for k, v in best_tn_params.items() if k not in ('lr','max_epochs')}
tn_full = TabNetClassifier(
    **tn_arch,
    optimizer_params={'lr': best_tn_params['lr']}
)
tn_full.fit(
    X_trval_prep, y_trval,
    max_epochs=best_tn_params['max_epochs']
)
joblib.dump(tn_full, 'saved_models/tabnet_full.pkl')


# 4) Re-fit CatBoostClassifier
from catboost import CatBoostClassifier

cb_full = CatBoostClassifier(**best_cb_params, verbose=0)
cb_full.fit(X_trval_prep, y_trval)
joblib.dump(cb_full, 'saved_models/catboost_full.pkl')


# 5) Re-fit LGBMClassifier
from lightgbm import LGBMClassifier

lgb_full = LGBMClassifier(**best_lgb_params)
lgb_full.fit(X_trval_prep, y_trval)
joblib.dump(lgb_full, 'saved_models/lgbm_full.pkl')

print("✅ Semua base‐models dan pipeline telah di-refit pada full train+val dan disimpan di `saved_models/`")

> Generate test-set meta-features

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# — Asumsi sudah tersedia di namespace —  
# X_test: DataFrame fitur test (OHLCV, sentimen, indikator, dsb.)  
# dist_full, preproc_full: transformer yang sudah di-fit  
# tt_full: instance SimpleTabTransformer yang sudah di-train penuh  
# best_tt_params, best_tn_params, best_cb_params, best_lgb_params, device  

# 1) Preprocessing test set
X_test_dist  = dist_full.transform(X_test.values)
X_test_prep  = preproc_full.transform(pd.DataFrame(X_test_dist, columns=X_test.columns))

# 2) Probabilitas dari tiap base-model
# a) TabTransformer
tt_full.eval()
with torch.no_grad():
    logits_tt_test = tt_full(torch.tensor(X_test_prep, dtype=torch.float32).to(device))
    probs_tt_test  = torch.softmax(logits_tt_test, dim=1)[:,1].cpu().numpy()

# b) TabNet
# (jika Anda punya tn_full yang sudah di-train, atau load dari file)
probs_tn_test = tn_full.predict_proba(X_test_prep)[:,1]

# c) CatBoost
probs_cb_test = cb_full.predict_proba(X_test_prep)[:,1]

# d) LightGBM
probs_lgb_test = lgb_full.predict_proba(X_test_prep)[:,1]

# 3) Bangun DataFrame meta-features untuk test set
df_meta_test = pd.DataFrame({
    'tt_oof':  probs_tt_test,
    'tn_oof':  probs_tn_test,
    'cb_oof':  probs_cb_test,
    'lgb_oof': probs_lgb_test,
}, index=X_test.index)

# Cek hasil sekilas
print(df_meta_test.head())
print("Shape meta-features test:", df_meta_test.shape)

> Final stacking prediction

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Dapatkan probabilitas meta-model pada test meta-features
meta_proba_test = meta_clf.predict_proba(df_meta_test)[:, 1]

# 2. Tentukan threshold
# Jika Anda menggunakan threshold optimal yang dihitung dari validasi (misal youden_j),
# ganti 0.5 di bawah dengan nilai itu. Contoh:
threshold = 0.5

# 3. Buat prediksi kelas akhir
preds_test = (meta_proba_test >= threshold).astype(int)

# 4. (Opsional) Evaluasi pada y_test jika tersedia
# Misal y_test_class adalah target 0/1 untuk test set
auc_test   = roc_auc_score(y_test_class, meta_proba_test)
acc_test   = accuracy_score(y_test_class, preds_test)
prec_test  = precision_score(y_test_class, preds_test)
rec_test   = recall_score(y_test_class, preds_test)
f1_test    = f1_score(y_test_class, preds_test)

print("Final Stacking Performance on Test Set:")
print(f"  AUC      : {auc_test:.4f}")
print(f"  Accuracy : {acc_test:.4f}")
print(f"  Precision: {prec_test:.4f}")
print(f"  Recall   : {rec_test:.4f}")
print(f"  F1-score : {f1_test:.4f}")

# 5. (Opsional) Simpan hasil prediksi
import pandas as pd
df_results = pd.DataFrame({
    'actual':    y_test_class,
    'proba_up':  meta_proba_test,
    'pred_up':   preds_test
}, index=X_test.index)

df_results.to_csv('saved_models/final_stack_predictions.csv', index=True)
print("Prediksi akhir telah disimpan ke 'saved_models/final_stack_predictions.csv'")

> Simpan hasil & evaluasi

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Pastikan direktori 'saved_models' ada
os.makedirs('saved_models', exist_ok=True)

# 1. Simpan meta-model untuk deployment
joblib.dump(meta_clf, 'saved_models/meta_model.pkl')

# 2. Buat DataFrame hasil prediksi
df_results = pd.DataFrame({
    'actual':   y_test_class,
    'proba_up': meta_proba_test,
    'pred_up':  preds_test
}, index=X_test.index)

# 3. Simpan ke CSV
df_results.to_csv('saved_models/final_stack_predictions.csv')
print("✅ Prediksi akhir disimpan di 'saved_models/final_stack_predictions.csv'")

# 4. Evaluasi metrik
auc   = roc_auc_score(y_test_class, df_results['proba_up'])
acc   = accuracy_score(y_test_class, df_results['pred_up'])
prec  = precision_score(y_test_class, df_results['pred_up'])
rec   = recall_score(y_test_class, df_results['pred_up'])
f1    = f1_score(y_test_class, df_results['pred_up'])

print("\n=== Final Evaluation on Test Set ===")
print(f"AUC      : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

# 5. (Opsional) Laporan detil
print("\nClassification Report:")
print(classification_report(y_test_class, df_results['pred_up']))

print("Confusion Matrix:")
print(confusion_matrix(y_test_class, df_results['pred_up']))

# Stacking Meta-Learner

> Eksperimen Beberapa Meta-Learner

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

# 1. Siapkan StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Kandidat meta-learner (tanpa MLP)
meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest':       RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced'),
    'LightGBM':           LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=5),
    'CatBoost':           CatBoostClassifier(iterations=100, learning_rate=0.05, depth=5, verbose=0)
}

# 3. Manual CV loop untuk AUC & F1
results = {}
for name, model in meta_models.items():
    aucs, f1s = [], []
    for train_idx, val_idx in cv.split(df_oof, y_trval):
        X_tr, X_val = df_oof.iloc[train_idx], df_oof.iloc[val_idx]
        y_tr, y_val = y_trval.iloc[train_idx], y_trval.iloc[val_idx]
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        aucs.append(roc_auc_score(y_val, proba))
        f1s.append(f1_score(y_val, pred))
    results[name] = {
        'mean_auc': np.mean(aucs),
        'std_auc':  np.std(aucs),
        'mean_f1':  np.mean(f1s),
        'std_f1':   np.std(f1s)
    }

# 4. Tampilkan perbandingan
df_meta_results = pd.DataFrame(results).T[['mean_auc','std_auc','mean_f1','std_f1']]
print(df_meta_results.sort_values('mean_auc', ascending=False))

> Siapkan Data Meta-Fitur (OOF)

In [ ]:
import pandas as pd

# Pastikan Anda sudah memiliki:
# oof_tt, oof_tn, oof_cb, oof_lgb  # keempat array OOF sepanjang X_trval
# X_trval.index                     # index kronologis train+val

# 1. Bangun DataFrame meta-fitur
df_oof = pd.DataFrame({
    'tt_oof' : oof_tt,   # probabilitas OOF dari TabTransformer
    'tn_oof' : oof_tn,   # probabilitas OOF dari TabNet
    'cb_oof' : oof_cb,   # probabilitas OOF dari CatBoost
    'lgb_oof': oof_lgb,  # probabilitas OOF dari LightGBM
}, index=X_trval.index)

# 2. Quick sanity checks
print(df_oof.head())
print("\nShape:", df_oof.shape)
print("Any NaNs in df_oof? ", df_oof.isna().any().any())
print("Value ranges per column:\n", df_oof.describe().T[['min','max']])

> Cross-Validation untuk Pemilihan Model

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

# 1. Siapkan StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Definisikan kandidat meta-learner
# (hapus MLPClassifier karena environment Anda sempat error)
meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest'       : RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced'),
    'LightGBM'           : LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=5),
    'CatBoost'           : CatBoostClassifier(iterations=100, learning_rate=0.05, depth=5, verbose=0)
}

# 3. Loop manual CV untuk tiap model
results = []
for name, model in meta_models.items():
    aucs, f1s = [], []
    for train_idx, val_idx in cv.split(df_oof, y_trval):
        X_tr, X_val = df_oof.iloc[train_idx], df_oof.iloc[val_idx]
        y_tr, y_val = y_trval.iloc[train_idx], y_trval.iloc[val_idx]
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        aucs.append(roc_auc_score(y_val, proba))
        f1s.append(f1_score(y_val, pred))
    results.append({
        'model': name,
        'mean_auc': np.mean(aucs),
        'std_auc' : np.std(aucs),
        'mean_f1' : np.mean(f1s),
        'std_f1'  : np.std(f1s)
    })

# 4. Ringkas dan tampilkan
df_meta_cv = pd.DataFrame(results).set_index('model')
print(df_meta_cv.sort_values('mean_auc', ascending=False))

> Bandingkan Performa & Pilih Meta-Learner Terbaik

In [ ]:
# df_meta_cv: index=model, kolom include 'mean_auc' dan 'mean_f1'
best_auc_model = df_meta_cv['mean_auc'].idxmax()
best_auc_score = df_meta_cv.loc[best_auc_model, 'mean_auc']
print(f"Meta-learner terbaik (AUC): {best_auc_model} (mean AUC = {best_auc_score:.4f})")

In [ ]:
best_f1_model = df_meta_cv['mean_f1'].idxmax()
best_f1_score = df_meta_cv.loc[best_f1_model, 'mean_f1']
print(f"Meta-learner terbaik (F1) : {best_f1_model} (mean F1 = {best_f1_score:.4f})")

In [ ]:
# Anda sebelumnya sudah punya dict meta_models
chosen_model = meta_models[best_auc_model]

> Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Ambil nama model terbaik berdasarkan mean AUC
best_auc_model = df_meta_cv['mean_auc'].idxmax()
print(f"→ Meta-learner terbaik menurut AUC: {best_auc_model}")

# 2. Ambil instance model-nya
chosen_model = meta_models[best_auc_model]

# 3. Siapkan grid hyperparameter sesuai jenis model
if isinstance(chosen_model, LogisticRegression):
    param_grid = {
        'C':       [0.01, 0.1, 1, 10],
        'penalty': ['l1','l2'],
        'solver':  ['liblinear']
    }
elif isinstance(chosen_model, LGBMClassifier):
    param_grid = {
        'n_estimators':   [100, 200, 400],
        'learning_rate':  [0.01, 0.05, 0.1],
        'max_depth':      [3, 5, 7],
        'reg_lambda':     [0, 1, 5]
    }
elif isinstance(chosen_model, CatBoostClassifier):
    param_grid = {
        'iterations':     [100, 200],
        'learning_rate':  [0.03, 0.05, 0.1],
        'depth':          [4, 6, 8]
    }
elif isinstance(chosen_model, RandomForestClassifier):
    param_grid = {
        'n_estimators': [100, 200, 400],
        'max_depth':    [3, 5, 7],
        'class_weight': ['balanced', None]
    }
else:
    raise ValueError(f"Tidak ada grid untuk model {best_auc_model!r}")

# 4. GridSearchCV dengan scoring AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=chosen_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=1,       # jalankan single‐threaded
    verbose=1
)
grid.fit(df_oof, y_trval)
print("Best AUC :", grid.best_score_)
print("Best params:", grid.best_params_)
best_meta = grid.best_estimator_

> Latih Meta-Learner Final

In [ ]:
import joblib
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# Asumsi:
# - best_meta: estimator hasil GridSearchCV (best_estimator_)
# - df_oof, y_trval: OOF features dan target train+val

# 1. Fit meta-learner pada seluruh OOF data
final_meta = best_meta
final_meta.fit(df_oof, y_trval)

# 2. Evaluasi pada OOF
proba_oof_meta = final_meta.predict_proba(df_oof)[:, 1]
pred_oof_meta  = (proba_oof_meta >= 0.5).astype(int)

metrics = {
    'AUC':    roc_auc_score(y_trval, proba_oof_meta),
    'Accuracy': accuracy_score(y_trval, pred_oof_meta),
    'Precision': precision_score(y_trval, pred_oof_meta),
    'Recall': recall_score(y_trval, pred_oof_meta),
    'F1-score': f1_score(y_trval, pred_oof_meta)
}

print("Final Meta-Learner Performance on OOF:")
for name, value in metrics.items():
    print(f"  {name:>9}: {value:.4f}")

# 3. Simpan model meta-learner
joblib.dump(final_meta, 'saved_models/final_meta_learner.pkl')
print("✅ Final meta-learner disimpan di 'saved_models/final_meta_learner.pkl'")

> Evaluasi OOF Meta-Model

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

# Asumsi:
# - final_meta: meta-learner yang sudah Anda latih ke seluruh df_oof & y_trval
# - df_oof, y_trval: DataFrame dan Series OOF-meta-features & target

# 1. Prediksi probabilitas & kelas
proba_oof = final_meta.predict_proba(df_oof)[:, 1]
pred_oof  = (proba_oof >= 0.5).astype(int)  # threshold 0.5, bisa disesuaikan

# 2. Hitung metrik
auc   = roc_auc_score(y_trval, proba_oof)
acc   = accuracy_score(y_trval, pred_oof)
prec  = precision_score(y_trval, pred_oof)
rec   = recall_score(y_trval, pred_oof)
f1    = f1_score(y_trval, pred_oof)

print("=== OOF Meta-Model Evaluation ===")
print(f"AUC      : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_trval, pred_oof))

print("Confusion Matrix:")
print(confusion_matrix(y_trval, pred_oof))

# 3. Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_trval, proba_oof)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"Meta (AUC={auc:.3f})")
plt.plot([0,1],[0,1],'k--', label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Meta-Model (OOF)")
plt.legend(loc="lower right")
plt.show()

> Kalibrasi Probabilitas 

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

# 1. Ambil probabilitas OOF meta‐model yang belum dikalibrasi
proba_uncal = final_meta.predict_proba(df_oof)[:, 1].reshape(-1, 1)

# 2. Fit LogisticRegression kecil sebagai Platt Scaling
platt = LogisticRegression(solver='lbfgs', max_iter=1000)
platt.fit(proba_uncal, y_trval)

# 3. Dapatkan probabilitas terkalibrasi pada OOF
proba_cal = platt.predict_proba(proba_uncal)[:, 1]

# 4. Ukur perbaikan kalibrasi
brier_uncal = brier_score_loss(y_trval, proba_uncal.ravel())
brier_cal   = brier_score_loss(y_trval, proba_cal)
auc_uncal   = roc_auc_score(y_trval, proba_uncal)
auc_cal     = roc_auc_score(y_trval, proba_cal)

print("— Manual Platt Scaling (OOF) —")
print(f"Brier score: uncal={brier_uncal:.4f}, cal={brier_cal:.4f}")
print(f"AUC        : uncal={auc_uncal:.4f}, cal={auc_cal:.4f}")

# 5. Simpan scaler
import joblib
joblib.dump(platt, 'saved_models/platt_scaler.pkl')
print("Platt scaler disimpan di 'saved_models/platt_scaler.pkl'")

# 6. Cara inferensi untuk test-set:
#    proba_test_uncal = final_meta.predict_proba(df_meta_test)[:,1].reshape(-1,1)
#    proba_test_cal   = platt.predict_proba(proba_test_uncal)[:,1]

> Deployment & Simpan

In [ ]:
import os
import joblib
import torch
import torch.nn as nn
import pandas as pd

# 1. Tentukan device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load transformers & pipelines
dist_tf    = joblib.load("saved_models/dist_transformer_full.pkl")
preproc_tf = joblib.load("saved_models/preprocessor_full.pkl")

# 3a. Load SimpleTabTransformer (class harus sudah didefinisikan di notebook ini)
tt = SimpleTabTransformer(
    input_dim=best_tt_params['input_dim'],
    embed_dim=best_tt_params['embed_dim'],
    n_heads=best_tt_params['n_heads'],
    n_layers=best_tt_params['n_layers'],
    dropout=best_tt_params['dropout']
).to(device)
tt.load_state_dict(torch.load("saved_models/tabtransformer_full.pth", map_location=device))
tt.eval()

# 3b. Load TabNet
tn = joblib.load("saved_models/tabnet_full.pkl")

# 3c. Load CatBoost
cb = joblib.load("saved_models/catboost_full.pkl")

# 3d. Load LightGBM
lgb = joblib.load("saved_models/lgbm_full.pkl")

# 4. Load meta-learner
final_meta = joblib.load("saved_models/final_meta_learner.pkl")

# 5. Optional: Platt scaler
platt_path = "saved_models/platt_scaler.pkl"
platt = joblib.load(platt_path) if os.path.exists(platt_path) else None

# 6. Define prediction function
def predict_direction(df_new: pd.DataFrame):
    # a) Transform data
    X_dist = dist_tf.transform(df_new.values)
    X_prep = preproc_tf.transform(pd.DataFrame(X_dist, columns=df_new.columns))

    # b) Base-model probabilities
    with torch.no_grad():
        logits = tt(torch.tensor(X_prep.values, dtype=torch.float32).to(device))
        proba_tt = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    proba_tn  = tn.predict_proba(X_prep)[:, 1]
    proba_cb  = cb.predict_proba(X_prep)[:, 1]
    proba_lgb = lgb.predict_proba(X_prep)[:, 1]

    # c) Meta-features DataFrame
    df_meta = pd.DataFrame({
        'tt_oof' : proba_tt,
        'tn_oof' : proba_tn,
        'cb_oof' : proba_cb,
        'lgb_oof': proba_lgb
    }, index=df_new.index)

    # d) Meta-model prediction
    proba_meta_uncal = final_meta.predict_proba(df_meta)[:, 1]
    proba_meta = platt.predict_proba(proba_meta_uncal.reshape(-1, 1))[:, 1] if platt else proba_meta_uncal

    # e) Thresholding
    preds = (proba_meta >= 0.5).astype(int)
    return preds, proba_meta

print("✅ Deployment ready. Fungsi `predict_direction(df_new)` siap digunakan.")


> Prediksi & Evaluasi Akhir pada Test

In [ ]:
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# — Asumsi sudah tersedia di namespace —
# df_meta_test : DataFrame meta-features test set
# y_test_class : Series target 0/1 untuk test set
# final_meta   : meta-learner terlatih penuh (best_meta)
# platt        : logistic scaler untuk kalibrasi (atau None)
# threshold    : threshold final (misal 0.5)

# 1. Prediksi probabilitas dan kelas
proba_test_uncal = final_meta.predict_proba(df_meta_test)[:, 1]
proba_test = (
    platt.predict_proba(proba_test_uncal.reshape(-1, 1))[:, 1]
    if platt is not None else proba_test_uncal
)
preds_test = (proba_test >= threshold).astype(int)

# 2. Hitung metrik utama
auc   = roc_auc_score(y_test_class, proba_test)
acc   = accuracy_score(y_test_class, preds_test)
prec  = precision_score(y_test_class, preds_test)
rec   = recall_score(y_test_class, preds_test)
f1    = f1_score(y_test_class, preds_test)

print("=== Final Evaluation on Test Set ===")
print(f"AUC      : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}\n")

# 3. Laporan detil
print("Classification Report:")
print(classification_report(y_test_class, preds_test))

print("Confusion Matrix:")
print(confusion_matrix(y_test_class, preds_test))

# 4. (Opsional) Simpan hasil ke CSV
df_results = pd.DataFrame({
    'actual'   : y_test_class,
    'proba_up' : proba_test,
    'pred_up'  : preds_test
}, index=df_meta_test.index)
df_results.to_csv('saved_models/final_test_predictions.csv')
print("\n✅ Final predictions saved to 'saved_models/final_test_predictions.csv'")

# Evaluasi & Visualisasi

> Evaluasi Kinerja Klasifikasi (target_class)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score
)

def evaluate_classification_from_df(models_dict, df, target_col='target_class',
                                    train_frac=0.70, val_frac=0.15):
    """
    1) Menggunakan DataFrame `df` yang sudah berisi fitur FE + kolom target.
    2) Otomatis split kronologis (train/val/test) berdasarkan proporsi.
    3) Hitung metrik klasifikasi & cetak DataFrame hasil.
    4) Plot AUC per-model, confusion matrix, ROC, PR curve, dan threshold analysis.
    
    Parameters
    ----------
    models_dict : dict
        Kamus nama_model -> model terlatih (harus support predict & predict_proba/decision_function).
    df : pd.DataFrame
        DataFrame berisi semua fitur teknikal, rolling, sentimen, dan kolom target.
    target_col : str
        Nama kolom target biner (default 'target_class').
    train_frac : float
        Proporsi data untuk train (default 0.70).
    val_frac : float
        Proporsi data untuk val setelah train (default 0.15).
    
    Returns
    -------
    pd.DataFrame
        Ringkasan metrik untuk setiap model.
    """
    # --- 1. Pisahkan fitur & target ---
    if target_col not in df.columns:
        raise ValueError(f"Kolom target '{target_col}' tidak ditemukan di DataFrame.")
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # --- 2. Split kronologis ---
    n = len(df)
    i_train = int(train_frac * n)
    i_val   = int((train_frac + val_frac) * n)
    X_test = X.iloc[i_val:]
    y_test = y.iloc[i_val:]
    
    # --- 3. Hitung metrik dasar ---
    records = []
    for name, mdl in models_dict.items():
        y_pred = mdl.predict(X_test)
        try:
            y_prob = mdl.predict_proba(X_test)[:, 1]
        except AttributeError:
            scores = mdl.decision_function(X_test)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min())
        records.append({
            'model':    name,
            'accuracy': accuracy_score(y_test, y_pred),
            'precision':precision_score(y_test, y_pred, zero_division=0),
            'recall':   recall_score(y_test, y_pred, zero_division=0),
            'f1':       f1_score(y_test, y_pred, zero_division=0),
            'auc':      roc_auc_score(y_test, y_prob)
        })
    df_res = pd.DataFrame(records).set_index('model')
    print(df_res, '\n')
    
    # --- 4. Plot AUC per-model ---
    plt.figure()
    df_res['auc'].sort_values().plot.barh(title='AUC-ROC per Model')
    plt.xlabel('AUC'); plt.tight_layout(); plt.show()
    
    # --- 5. Evaluasi detail untuk model terbaik ---
    best = df_res['auc'].idxmax()
    mdl_best = models_dict[best]
    y_pred_b = mdl_best.predict(X_test)
    try:
        y_prob_b = mdl_best.predict_proba(X_test)[:,1]
    except AttributeError:
        scores = mdl_best.decision_function(X_test)
        y_prob_b = (scores - scores.min()) / (scores.max() - scores.min())
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_b)
    plt.figure(figsize=(4,4))
    plt.imshow(cm, cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix — {best}')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i,j], ha='center', va='center')
    plt.xticks([0,1]); plt.yticks([0,1]); plt.tight_layout(); plt.show()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob_b)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {df_res.loc[best,"auc"]:.3f}')
    plt.plot([0,1],[0,1],'--')
    plt.title(f'ROC Curve — {best}')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.tight_layout(); plt.show()
    
    # Precision–Recall Curve
    prec, rec, _ = precision_recall_curve(y_test, y_prob_b)
    ap = average_precision_score(y_test, y_prob_b)
    plt.figure()
    plt.plot(rec, prec, label=f'AP = {ap:.3f}')
    plt.title(f'Precision–Recall — {best}')
    plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.tight_layout(); plt.show()
    
    # Threshold Analysis
    thr = np.linspace(0,1,101)
    f1s, youdens = [], []
    for t in thr:
        p = (y_prob_b >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
        f1s.append(f1_score(y_test, p, zero_division=0))
        youdens.append(tp/(tp+fn) - fp/(fp+tn))
    best_f1 = thr[np.argmax(f1s)]
    best_j  = thr[np.argmax(youdens)]
    print(f"Best F1 thresh  : {best_f1:.2f}")
    print(f"Best Youden J   : {best_j:.2f}")
    plt.figure()
    plt.plot(thr, f1s, label='F1 Score')
    plt.plot(thr, youdens, label="Youden's J")
    plt.title(f'Threshold Analysis — {best}')
    plt.xlabel('Threshold'); plt.legend(); plt.tight_layout(); plt.show()
    
    return df_res

# Contoh penggunaan:
# df_results = evaluate_classification_from_df(models_dict, df_full, 'target_class')

> Evaluasi Kinerja Regresi (target_reg)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_regression_from_df(models_dict, df, target_col='target_reg',
                                train_frac=0.70, val_frac=0.15):
    """
    1) Menggunakan DataFrame `df` yang sudah berisi fitur FE + kolom target_reg.
    2) Otomatis split kronologis (train/val/test) berdasarkan proporsi.
    3) Hitung metrik regresi: RMSE, MAE, R².
    4) Plot:
       - Scatter actual vs predicted (garis y=x)
       - Histogram residuals
       - Scatter residual vs fitted values
    """
    # --- 1. Pisahkan fitur & target ---
    if target_col not in df.columns:
        raise ValueError(f"Kolom target '{target_col}' tidak ditemukan di DataFrame.")
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # --- 2. Split kronologis ---
    n = len(df)
    i_train = int(train_frac * n)
    i_val   = int((train_frac + val_frac) * n)
    X_test = X.iloc[i_val:]
    y_test = y.iloc[i_val:]
    
    # --- 3. Hitung metrik untuk setiap model ---
    records = []
    for name, mdl in models_dict.items():
        y_pred = mdl.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        records.append({
            'model': name,
            'RMSE':  rmse,
            'MAE':   mae,
            'R2':    r2
        })
    df_res = pd.DataFrame(records).set_index('model')
    print(df_res, '\n')
    
    # --- 4a. Bar chart RMSE per-model ---
    plt.figure()
    df_res['RMSE'].sort_values().plot.barh(title='RMSE per Model')
    plt.xlabel('RMSE'); plt.tight_layout(); plt.show()
    
    # --- Pilih model terbaik (berdasarkan RMSE terendah) ---
    best = df_res['RMSE'].idxmin()
    mdl_best = models_dict[best]
    y_pred_b = mdl_best.predict(X_test)
    
    # --- 4b. Scatter actual vs predicted ---
    plt.figure()
    plt.scatter(y_test, y_pred_b)
    lims = [min(y_test.min(), y_pred_b.min()), max(y_test.max(), y_pred_b.max())]
    plt.plot(lims, lims)
    plt.title(f'Actual vs Predicted — {best}')
    plt.xlabel('Actual log-return'); plt.ylabel('Predicted log-return')
    plt.tight_layout(); plt.show()
    
    # --- 4c. Histogram residuals ---
    residuals = y_test - y_pred_b
    plt.figure()
    plt.hist(residuals, bins=30, density=True)
    plt.title(f'Residuals Distribution — {best}')
    plt.xlabel('Residual'); plt.ylabel('Density')
    plt.tight_layout(); plt.show()
    
    # --- 4d. Residuals vs Fitted ---
    plt.figure()
    plt.scatter(y_pred_b, residuals)
    plt.axhline(0, linestyle='--')
    plt.title(f'Residuals vs Fitted — {best}')
    plt.xlabel('Fitted values'); plt.ylabel('Residuals')
    plt.tight_layout(); plt.show()
    
    return df_res

# Contoh penggunaan:
# df_reg_results = evaluate_regression_from_df(models_dict, df_full, 'target_reg')

> Perbandingan Base-Model vs. Stacking

In [ ]:
# 0. Pastikan X_train, X_val, y_train_class, y_val_class
#    semuanya menyimpan datetime-index yang sama seperti saat split.

# 1. Alignment berdasarkan index
X_tr = X_train.align(y_train_class, join='inner', axis=0)[0]
y_tr = y_train_class.align(X_train,        join='inner', axis=0)[0]

X_va = X_val.align(y_val_class, join='inner', axis=0)[0]
y_va = y_val_class.align(X_val,  join='inner', axis=0)[0]

# 2. Sekarang reset index dan concat
X_tr = X_tr.reset_index(drop=True)
y_tr = y_tr.reset_index(drop=True)
X_va = X_va.reset_index(drop=True)
y_va = y_va.reset_index(drop=True)

X_trval = pd.concat([X_tr, X_va], axis=0).reset_index(drop=True)
y_trval = pd.concat([y_tr, y_va], axis=0).reset_index(drop=True)

# Verifikasi ulang
assert len(X_trval) == len(y_trval), f"Masih mismatch: {len(X_trval)} vs {len(y_trval)}"

In [ ]:
# 1. Definisikan semua fitur asli tanpa kolom stacking
stack_oof_cols = ['tt_oof', 'tn_oof', 'cb_oof', 'lgb_oof']
all_features = [col for col in X_trval.columns if col not in stack_oof_cols]

# 2. Retrain base‐models pada all_features
X_trval_full = X_trval[all_features]
y_trval_full = y_trval

cb_base.fit(X_trval_full, y_trval_full)
lgb_base.fit(X_trval_full, y_trval_full)
# TabNet & TabTransformer juga pakai all_features

# 3. Hitung probabilitas pada X_test, pilih same all_features
X_te_full = X_te[all_features]

probs_cb  = cb_base.predict_proba(X_te_full)[:,1]
probs_lgb = lgb_base.predict_proba(X_te_full)[:,1]
probs_tn = tn.predict_proba(X_te_full.values)[:,1]

with torch.no_grad():
    logits_tt = tt(torch.tensor(X_te_full.values, dtype=torch.float32).to(device))
    probs_tt  = torch.softmax(logits_tt, dim=1)[:,1].cpu().numpy()

# 4. Stacking & perbandingan
meta_test = np.vstack([probs_tt, probs_tn, probs_cb, probs_lgb]).T
probs_ens = meta_clf.predict_proba(meta_test)[:,1]

# Pastikan panjang sama
assert len(y_te)==len(probs_tt)==len(probs_tn)==len(probs_cb)==len(probs_lgb)==len(probs_ens)

# 5. Buat tabel perbandingan metrik
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

models = {
    'TabTransformer': probs_tt,
    'TabNet'        : probs_tn,
    'CatBoost'      : probs_cb,
    'LightGBM'      : probs_lgb,
    'Stacking'      : probs_ens
}
thresholds = {m: 0.5 for m in models}
thresholds['Stacking'] = best_thr_f1

rows = {}
for name, probs in models.items():
    preds = (probs >= thresholds[name]).astype(int)
    rows[name] = {
        'AUC'      : roc_auc_score(y_te, probs),
        'Accuracy' : accuracy_score(y_te, preds),
        'Precision': precision_score(y_te, preds, zero_division=0),
        'Recall'   : recall_score(y_te, preds),
        'F1'       : f1_score(y_te, preds)
    }

df_comparison = pd.DataFrame(rows).T
print("=== Perbandingan Base‐Models vs Stacking Ensemble ===")
print(df_comparison)

> Analisis Kalibrasi Probabilitas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    roc_curve
)

# 1. Ambil probabilitas ensemble (sebelum kalibrasi) di validation set
#    (ini yang sudah Anda hitung saat stacking)
probs_val_ens = meta_clf.predict_proba(meta_train)[:,1]

# 2. Fit LogisticRegression untuk Platt Scaling
platt = LogisticRegression(solver='lbfgs')
platt.fit(probs_val_ens.reshape(-1,1), y_val_class)

# 3. Terapkan ke test set
probs_test_ens_uncal = probs_ens  # probabilitas ensemble original di X_test
probs_test_ens_cal   = platt.predict_proba(probs_test_ens_uncal.reshape(-1,1))[:,1]

# 4. Plot calibration curve manual
def calibration_curve_manual(y_true, prob_pred, n_bins=10):
    bins    = np.linspace(0,1,n_bins+1)
    bin_ids = np.digitize(prob_pred, bins) - 1
    mean_pred, frac_pos = [], []
    for i in range(n_bins):
        mask = bin_ids == i
        if mask.sum()>0:
            mean_pred.append(prob_pred[mask].mean())
            frac_pos.append(y_true[mask].mean())
        else:
            mean = (bins[i]+bins[i+1])/2
            mean_pred.append(mean)
            frac_pos.append(np.nan)
    return np.array(mean_pred), np.array(frac_pos)

mean_uncal, frac_uncal = calibration_curve_manual(y_test_class, probs_test_ens_uncal)
mean_cal,   frac_cal   = calibration_curve_manual(y_test_class, probs_test_ens_cal)

plt.figure(figsize=(6,4))
plt.plot(mean_uncal, frac_uncal, 's-', label='Uncalibrated')
plt.plot(mean_cal,   frac_cal,   'o-', label='Calibrated (Platt)')
plt.plot([0,1],[0,1],'k--', label='Perfectly calibrated')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve — Stacking Ensemble')
plt.legend()
plt.tight_layout()
plt.show()

# 5. Threshold optimization on calibrated probabilities
#    misalnya maksimalkan F1 di validation:
prec, rec, thr_pr = precision_recall_curve(y_val_class, platt.predict_proba(probs_val_ens.reshape(-1,1))[:,1])
f1_scores = 2*(prec[:-1]*rec[:-1])/(prec[:-1]+rec[:-1]+1e-12)
best_idx  = np.argmax(f1_scores)
best_thr_f1_ens = thr_pr[best_idx]
print(f"Optimal threshold (max F1) for ensemble: {best_thr_f1_ens:.3f}, F1: {f1_scores[best_idx]:.3f}")

# 6. Evaluasi akhir pada test set dengan calibrated probs & threshold
from sklearn.metrics import classification_report
preds_test_cal = (probs_test_ens_cal >= best_thr_f1_ens).astype(int)
print("\nClassification Report — Stacking Ensemble (Calibrated & Thresholded):")
print(classification_report(
    y_test_class, 
    preds_test_cal,
    target_names=['Down (0)','Up (1)'],
    digits=4
))

> Interpretabilitas Fitur

In [ ]:
import pandas as pd
import numpy as np

# 1.1 Nama fitur meta sesuai urutan saat Anda membentuk meta_train: 
#     misalnya ['TabTransformer', 'TabNet', 'CatBoost', 'LightGBM']
meta_features_names = ['TabTransformer', 'TabNet', 'CatBoost', 'LightGBM']

# 1.2 Ambil koefisien (LogisticRegression memiliki satu array, shape=(1,4))
coefs = meta_clf.coef_[0]

# 1.3 Buat DataFrame untuk mempermudah interpretasi
df_meta_imp = pd.DataFrame({
    'Model' : meta_features_names,
    'Weight': coefs
}).sort_values(by='Weight', ascending=False).reset_index(drop=True)

print("=== Meta‐Model Weights (Logistic Regression) ===")
print(df_meta_imp)

In [ ]:
# 2.1 CatBoost
fi_cb = pd.Series(
    cb_base.get_feature_importance(type='FeatureImportance'),
    index=base_features
).sort_values(ascending=False).head(10).rename("CatBoost")

# 2.2 LightGBM
fi_lgb = pd.Series(
    lgb_base.feature_importances_,
    index=base_features
).sort_values(ascending=False).head(10).rename("LightGBM")

# 2.3 TabNet
fi_tn = pd.Series(
    tn.feature_importances_,
    index=base_features
).sort_values(ascending=False).head(10).rename("TabNet")

# 2.4 Simple TabTransformer (drop‐column)
#    Asumsikan Anda sudah punya importances dict dari drop‐column
#    importances[col] = ΔAUC
fi_tt = pd.Series(importances).sort_values(ascending=False).head(10).rename("TabTransformer")

# 2.5 Gabungkan ke satu tabel
df_feat_imp = pd.concat([fi_cb, fi_lgb, fi_tn, fi_tt], axis=1)
print("=== Top-10 Feature Importance per Base‐Model ===")
print(df_feat_imp)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# 3.1 Siapkan data surrogate: meta_train & y_val_class
surrogate = DecisionTreeClassifier(max_depth=3)
surrogate.fit(meta_train, y_val_class)

# 3.2 Cetak aturan pohon
rules = export_text(surrogate, feature_names=meta_features_names)
print("=== Surrogate Decision Tree Rules ===")
print(rules)

> Visualisasi Time-Series

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Siapkan DataFrame hasil inference lengkap dengan timestamp
#    Asumsikan X_test semula memiliki DateTimeIndex atau kolom 'date'
df_vis = pd.DataFrame({
    'date'       : X_test.index,     # atau X_test['date'] jika kolom
    'Close'      : df_daily.loc[X_test.index, 'Close'],  # harga aktual
    'prob_up'    : probs_ens,        # probabilitas ensemble “Up”
    'pred_up'    : preds_ens         # prediksi biner (0/1)
})
df_vis = df_vis.set_index('date')

# 2. Plot harga + probabilitas
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(df_vis.index, df_vis['Close'], color='tab:blue', label='Close Price')
ax1.set_ylabel('Price (USD)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(df_vis.index, df_vis['prob_up'], color='tab:orange', linestyle='--', label='Prob Up')
ax2.set_ylabel('Prob. Up', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

# Legends
lines, labels = ax1.get_legend_handles_labels()
l2, l2l    = ax2.get_legend_handles_labels()
ax1.legend(lines+l2, labels+l2l, loc='upper left')
ax1.set_title('Bitcoin Close Price vs Ensemble Prob. Up')
plt.tight_layout()
plt.show()

# 3. Plot sinyal prediksi pada price chart
plt.figure(figsize=(10,4))
plt.plot(df_vis.index, df_vis['Close'], label='Close Price', color='black')
# Tandai titik buy (pred_up==1 pada hari sebelumnya, artinya besok naik)
buy_signals  = df_vis[df_vis['pred_up'] == 1].index
sell_signals = df_vis[df_vis['pred_up'] == 0].index
plt.scatter(buy_signals, df_vis.loc[buy_signals,'Close'], marker='^', color='green', label='Predicted Up', s=50)
plt.scatter(sell_signals, df_vis.loc[sell_signals,'Close'], marker='v', color='red',   label='Predicted Down', s=50)

plt.title('Prediksi Arah Harga Bitcoin (Ensemble)')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

# 4. Rolling performance (win rate)
df_vis['correct'] = (df_vis['pred_up'] == (df_vis['Close'].shift(-1) > df_vis['Close']).astype(int)).astype(int)
rolling_acc = df_vis['correct'].rolling(window=30).mean()

plt.figure(figsize=(10,3))
plt.plot(rolling_acc.index, rolling_acc, label='30-day Rolling Accuracy')
plt.axhline(0.5, color='gray', linestyle='--')
plt.fill_between(rolling_acc.index, rolling_acc, 0.5, where=(rolling_acc>=0.5), color='green', alpha=0.2)
plt.fill_between(rolling_acc.index, rolling_acc, 0.5, where=(rolling_acc<0.5),  color='red',   alpha=0.2)
plt.title('Rolling 30-Day Prediction Accuracy')
plt.ylabel('Accuracy')
plt.ylim(0,1)
plt.legend()
plt.tight_layout()
plt.show()

> Diagnostik Outlier & Distribusi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore

# Asumsi: df_daily, probs_ens, preds_ens sudah ada; index X_test sesuai df_daily
# 1. Bangun DataFrame diagnostic
df_diag = pd.DataFrame({
    'Close': df_daily.loc[X_test.index, 'Close'],
    'prob_up': probs_ens,
    'pred_up': preds_ens
})
df_diag['actual'] = (df_diag['Close'].shift(-1) > df_diag['Close']).astype(int)
df_diag = df_diag.dropna()

# 2. Hitung residual = prob_up - actual
df_diag['residual'] = df_diag['prob_up'] - df_diag['actual']

# Boxplot residual
plt.figure(figsize=(6,4))
plt.boxplot(df_diag['residual'], vert=False)
plt.title('Boxplot of Prediction Residuals')
plt.xlabel('Residual (prob_up - actual)')
plt.tight_layout()
plt.show()

# Histogram residual
plt.figure(figsize=(6,4))
plt.hist(df_diag['residual'], bins=30, edgecolor='k')
plt.title('Histogram of Prediction Residuals')
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Identifikasi outlier via z-score
df_diag['res_z'] = zscore(df_diag['residual'])
outliers = df_diag[np.abs(df_diag['res_z']) > 3]
print(f"Number of residual outliers: {len(outliers)}")
print(outliers.head()[['Close','prob_up','actual','residual','res_z']])

# Distribusi returns pada df_daily
plt.figure(figsize=(6,4))
plt.hist(df_daily['return'].dropna(), bins=50, edgecolor='k')
plt.title('Distribution of Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.boxplot(df_daily['return'].dropna(), vert=False)
plt.title('Boxplot of Daily Returns')
plt.xlabel('Daily Return')
plt.tight_layout()
plt.show()

> Volume & Sentimen

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore

# Asumsi: df_daily memiliki kolom 'Volume' dan 'sentiment'

# 1. Histogram & Boxplot Volume
plt.figure(figsize=(6,3))
plt.hist(df_daily['Volume'].dropna(), bins=50, edgecolor='k')
plt.title('Distribution of Daily Volume')
plt.xlabel('Volume')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,3))
plt.boxplot(df_daily['Volume'].dropna(), vert=False)
plt.title('Boxplot of Daily Volume')
plt.xlabel('Volume')
plt.tight_layout()
plt.show()

# 2. Outlier detection on Volume via Z-score
vol_z = zscore(df_daily['Volume'].dropna())
out_vol = df_daily['Volume'].dropna()[np.abs(vol_z) > 3]
print(f"Number of volume outliers (|z|>3): {len(out_vol)}")
print(out_vol.head())

# 3. Histogram & Boxplot Sentiment
plt.figure(figsize=(6,3))
plt.hist(df_daily['sentiment'].dropna(), bins=30, edgecolor='k')
plt.title('Distribution of Daily Average Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,3))
plt.boxplot(df_daily['sentiment'].dropna(), vert=False)
plt.title('Boxplot of Daily Average Sentiment')
plt.xlabel('Sentiment')
plt.tight_layout()
plt.show()

# 4. Outlier detection on Sentiment via Z-score
sent_z = zscore(df_daily['sentiment'].dropna())
out_sent = df_daily['sentiment'].dropna()[np.abs(sent_z) > 3]
print(f"Number of sentiment outliers (|z|>3): {len(out_sent)}")
print(out_sent.head())

# Inferensi & Deployment

> Bundle & Load Pipeline

In [ ]:
import os
import joblib
import torch
import numpy as np


def bundle_pipeline(model_dir: str, bundle_path: str) -> None:
    """
    Load individual pipeline artifacts from `model_dir` dynamically and bundle them into a single file at `bundle_path`.
    """
    artifacts = {}
    files = os.listdir(model_dir)

    def find_file(keywords, exts=None):
        if isinstance(keywords, str):
            keywords = [keywords]
        if exts is None:
            exts = ['.pkl', '.joblib']
        for fname in files:
            lname = fname.lower()
            for keyword in keywords:
                if keyword.lower() in lname:
                    for ext in exts:
                        if lname.endswith(ext):
                            return os.path.join(model_dir, fname)
        raise FileNotFoundError(f"No file found for keywords {keywords} in {model_dir}")

    # Load preprocessing objects
    artifacts['preprocessor'] = joblib.load(find_file(['preprocessor', 'processor']))
    # Try multiple variants for distribution transformer
    artifacts['dist_transformer'] = joblib.load(
        find_file(['distribution_transformer', 'dist_transformer', 'transformer'])
    )

    # Load base models (sklearn, CatBoost, LightGBM)
    base_models = {}
    for fname in files:
        lname = fname.lower()
        if lname.endswith('_model.pkl') or lname.endswith('_model.joblib'):
            name = os.path.splitext(fname)[0].replace('_model', '')
            base_models[name] = joblib.load(os.path.join(model_dir, fname))

    # Load PyTorch-based models (.pt)
    for fname in files:
        if fname.lower().endswith('.pt'):
            name = os.path.splitext(fname)[0]
            base_models[name] = torch.load(os.path.join(model_dir, fname), map_location='cpu')

    artifacts['base_models'] = base_models

    # Load meta-model, accepting variants
    artifacts['meta_clf'] = joblib.load(
        find_file(['meta', 'meta_clf', 'stacking'])
    )

    # Serialize the bundled pipeline
    joblib.dump(artifacts, bundle_path)
    print(f"Pipeline bundle saved to: {bundle_path}")



def load_pipeline(bundle_path: str) -> dict:
    """
    Load a bundled pipeline from `bundle_path` and return its components as a dict.
    """
    return joblib.load(bundle_path)


class PipelinePredictor:
    """
    Wrapper class to perform inference using a bundled pipeline.
    """
    def __init__(self, bundle: dict):
        self.preprocessor = bundle['preprocessor']
        self.dist_transformer = bundle['dist_transformer']
        self.base_models = bundle['base_models']
        self.meta_clf = bundle['meta_clf']

    def predict(self, df_new, threshold: float = 0.5):
        """
        Given a raw DataFrame `df_new`, preprocess, compute base-model probabilities,
        then predict final class and probability via the meta-classifier.

        Parameters:
        - df_new: pd.DataFrame, raw features
        - threshold: float, decision threshold for binary classification

        Returns:
        - final_pred: np.ndarray of ints (0/1)
        - final_prob: np.ndarray of floats
        """
        # Step 1: preprocessing
        X_proc = self.preprocessor.transform(df_new)
        X_trans = self.dist_transformer.transform(X_proc)

        # Step 2: collect base-model probabilities
        prob_list = []
        for name, model in self.base_models.items():
            if hasattr(model, 'predict_proba'):
                probs = model.predict_proba(X_trans)[:, 1]
            else:
                tensor_in = torch.tensor(X_trans, dtype=torch.float32)
                # assume model outputs probabilities via predict_proba
                probs = model.predict_proba(tensor_in).detach().cpu().numpy()[:, 1]
            prob_list.append(probs.reshape(-1, 1))

        meta_features = np.concatenate(prob_list, axis=1)

        # Step 3: meta-classifier inference
        final_prob = self.meta_clf.predict_proba(meta_features)[:, 1]
        final_pred = (final_prob >= threshold).astype(int)
        return final_pred, final_prob


# Example usage
if __name__ == '__main__':
    MODEL_DIR = 'saved_models'
    BUNDLE_PATH = os.path.join(MODEL_DIR, 'pipeline_bundle.pkl')

    # Bundle and save
    try:
        bundle_pipeline(MODEL_DIR, BUNDLE_PATH)
    except FileNotFoundError as e:
        print(f"Error bundling pipeline: {e}")

    # Load bundle
    bundle = load_pipeline(BUNDLE_PATH)
    predictor = PipelinePredictor(bundle)

    # df_new = ...  # load or prepare new DataFrame
    # preds, probs = predictor.predict(df_new, threshold=0.5)
    # print(preds, probs)

> Batch Inference

In [ ]:
import os
import sys
import joblib
import torch
import numpy as np
import pandas as pd
import logging

# Configure logging
type = logging.INFO
logging.basicConfig(
    format='%(asctime)s %(levelname)-8s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

def bundle_pipeline(model_dir: str, bundle_path: str) -> None:
    """
    Bundle preprocessing transforms, base models, and meta-model into a single file.
    """
    logger.info(f"Bundling pipeline from '{model_dir}' to '{bundle_path}'")
    artifacts = {}
    files = os.listdir(model_dir)

    def find_file(keywords, exts=None):
        if isinstance(keywords, str):
            keywords = [keywords]
        exts = exts or ['.pkl', '.joblib']
        for fname in files:
            lname = fname.lower()
            if any(kw.lower() in lname for kw in keywords):
                if any(lname.endswith(ext) for ext in exts):
                    return os.path.join(model_dir, fname)
        raise FileNotFoundError(f"Missing artifact for {keywords}")

    # Load transforms
    artifacts['preprocessor'] = joblib.load(find_file(['preprocessor', 'processor']))
    artifacts['dist_transformer'] = joblib.load(find_file(['distribution_transformer', 'transformer']))

    # Load base models\ n    artifacts['base_models'] = {}
    for fname in files:
        path = os.path.join(model_dir, fname)
        if fname.lower().endswith(('_model.pkl', '_model.joblib')):
            key = fname.replace('_model.pkl', '').replace('_model.joblib', '')
            artifacts['base_models'][key] = joblib.load(path)
        elif fname.lower().endswith('.pt'):
            key = os.path.splitext(fname)[0]
            artifacts['base_models'][key] = torch.load(path, map_location='cpu')
    if not artifacts['base_models']:
        raise FileNotFoundError("No base-model files found.")

    # Load meta-model
    artifacts['meta_clf'] = joblib.load(find_file(['meta_clf', 'stacking']))

    # Save bundled pipeline
    os.makedirs(os.path.dirname(bundle_path) or '.', exist_ok=True)
    joblib.dump(artifacts, bundle_path)
    logger.info(f"Pipeline bundle saved at '{bundle_path}'")


def load_pipeline(bundle_path: str) -> dict:
    """
    Load bundled pipeline and return components dictionary.
    """
    if not os.path.exists(bundle_path):
        raise FileNotFoundError(f"Bundle not found: {bundle_path}")
    return joblib.load(bundle_path)


class PipelinePredictor:
    """
    Inference wrapper for a bundled pipeline.
    """
    def __init__(self, bundle: dict):
        self.preprocessor = bundle['preprocessor']
        self.dist_transformer = bundle['dist_transformer']
        self.base_models = bundle['base_models']
        self.meta_clf = bundle['meta_clf']

    def predict(self, df: pd.DataFrame, threshold: float = 0.5):
        # Preprocess
        X_proc = self.preprocessor.transform(df)
        X_trans = self.dist_transformer.transform(X_proc)
        # Base predictions
        prob_list = []
        for name, model in self.base_models.items():
            if hasattr(model, 'predict_proba'):
                p = model.predict_proba(X_trans)[:, 1]
            else:
                tensor = torch.tensor(X_trans, dtype=torch.float32)
                p = model.predict_proba(tensor).detach().cpu().numpy()[:, 1]
            prob_list.append(p.reshape(-1, 1))
        meta_feat = np.hstack(prob_list)
        # Meta-model
        final_prob = self.meta_clf.predict_proba(meta_feat)[:, 1]
        final_pred = (final_prob >= threshold).astype(int)
        return final_pred, final_prob


def run_batch_inference(
    df_or_path,
    bundle_path: str,
    model_dir: str = None,
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Run batch inference given DataFrame or file path. Returns DataFrame with predictions.
    """
    # Load data
    if isinstance(df_or_path, pd.DataFrame):
        df = df_or_path.copy()
    else:
        ext = os.path.splitext(df_or_path)[1].lower()
        if ext == '.csv':
            df = pd.read_csv(df_or_path)
        elif ext in ('.parquet', '.pq'):
            df = pd.read_parquet(df_or_path)
        else:
            raise ValueError("Unsupported format: CSV or Parquet")

    # Ensure bundle
    if model_dir and not os.path.exists(bundle_path):
        bundle_pipeline(model_dir, bundle_path)

    bundle = load_pipeline(bundle_path)
    predictor = PipelinePredictor(bundle)
    preds, probs = predictor.predict(df, threshold)
    df['prediction'] = preds
    df['probability'] = probs
    return df

# Usage example:
#
# from pipeline_bundle_load import run_batch_inference
# df_results = run_batch_inference('data/my_input.parquet', 'saved_models/pipeline_bundle.pkl', model_dir='saved_models')
# df_results.to_csv('data/results_with_preds.csv', index=False)


> Deployment / Production

In [ ]:
import os
import sys
import joblib
import torch
import numpy as np
import pandas as pd
import logging

# Configure logging
type = logging.INFO
logging.basicConfig(
    format='%(asctime)s %(levelname)-8s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

def bundle_pipeline(model_dir: str, bundle_path: str) -> None:
    """
    Bundle preprocessing transforms, base models, and meta-model into a single file.
    """
    logger.info(f"Bundling pipeline from '{model_dir}' to '{bundle_path}'")
    artifacts = {}
    files = os.listdir(model_dir)

    def find_file(keywords, exts=None):
        if isinstance(keywords, str):
            keywords = [keywords]
        exts = exts or ['.pkl', '.joblib']
        for fname in files:
            lname = fname.lower()
            if any(kw.lower() in lname for kw in keywords):
                if any(lname.endswith(ext) for ext in exts):
                    return os.path.join(model_dir, fname)
        raise FileNotFoundError(f"Missing artifact for {keywords}")

    # Load transforms
    artifacts['preprocessor'] = joblib.load(find_file(['preprocessor', 'processor']))
    artifacts['dist_transformer'] = joblib.load(find_file(['distribution_transformer', 'transformer']))

    # Load base models\ n    artifacts['base_models'] = {}
    for fname in files:
        path = os.path.join(model_dir, fname)
        if fname.lower().endswith(('_model.pkl', '_model.joblib')):
            key = fname.replace('_model.pkl', '').replace('_model.joblib', '')
            artifacts['base_models'][key] = joblib.load(path)
        elif fname.lower().endswith('.pt'):
            key = os.path.splitext(fname)[0]
            artifacts['base_models'][key] = torch.load(path, map_location='cpu')
    if not artifacts['base_models']:
        raise FileNotFoundError("No base-model files found.")

    # Load meta-model
    artifacts['meta_clf'] = joblib.load(find_file(['meta_clf', 'stacking']))

    # Save bundled pipeline
    os.makedirs(os.path.dirname(bundle_path) or '.', exist_ok=True)
    joblib.dump(artifacts, bundle_path)
    logger.info(f"Pipeline bundle saved at '{bundle_path}'")


def load_pipeline(bundle_path: str) -> dict:
    """
    Load bundled pipeline and return components dictionary.
    """
    if not os.path.exists(bundle_path):
        raise FileNotFoundError(f"Bundle not found: {bundle_path}")
    return joblib.load(bundle_path)


class PipelinePredictor:
    """
    Inference wrapper for a bundled pipeline.
    """
    def __init__(self, bundle: dict):
        self.preprocessor = bundle['preprocessor']
        self.dist_transformer = bundle['dist_transformer']
        self.base_models = bundle['base_models']
        self.meta_clf = bundle['meta_clf']

    def predict(self, df: pd.DataFrame, threshold: float = 0.5):
        # Preprocess
        X_proc = self.preprocessor.transform(df)
        X_trans = self.dist_transformer.transform(X_proc)
        # Base predictions
        prob_list = []
        for name, model in self.base_models.items():
            if hasattr(model, 'predict_proba'):
                p = model.predict_proba(X_trans)[:, 1]
            else:
                tensor = torch.tensor(X_trans, dtype=torch.float32)
                p = model.predict_proba(tensor).detach().cpu().numpy()[:, 1]
            prob_list.append(p.reshape(-1, 1))
        meta_feat = np.hstack(prob_list)
        # Meta-model
        final_prob = self.meta_clf.predict_proba(meta_feat)[:, 1]
        final_pred = (final_prob >= threshold).astype(int)
        return final_pred, final_prob


def run_batch_inference(
    df_or_path,
    bundle_path: str,
    model_dir: str = None,
    threshold: float = 0.5
) -> pd.DataFrame:
    """
    Run batch inference given DataFrame or file path. Returns DataFrame with predictions.
    """
    # Load data
    if isinstance(df_or_path, pd.DataFrame):
        df = df_or_path.copy()
    else:
        ext = os.path.splitext(df_or_path)[1].lower()
        if ext == '.csv':
            df = pd.read_csv(df_or_path)
        elif ext in ('.parquet', '.pq'):
            df = pd.read_parquet(df_or_path)
        else:
            raise ValueError("Unsupported format: CSV or Parquet")

    # Ensure bundle
    if model_dir and not os.path.exists(bundle_path):
        bundle_pipeline(model_dir, bundle_path)

    bundle = load_pipeline(bundle_path)
    predictor = PipelinePredictor(bundle)
    preds, probs = predictor.predict(df, threshold)
    df['prediction'] = preds
    df['probability'] = probs
    return df

# Usage example:
#
# from pipeline_bundle_load import run_batch_inference
# df_results = run_batch_inference('data/my_input.parquet', 'saved_models/pipeline_bundle.pkl', model_dir='saved_models')
# df_results.to_csv('data/results_with_preds.csv', index=False)

In [ ]:
#from predict import run_batch_inference

# Jika ingin pakai DataFrame langsung:
#import pandas as pd
#df_new = pd.read_csv('data/new_data.csv')

# Jalankan inferensi
#df_out = run_batch_inference(
    #df_or_path=df_new,
    #bundle_path='saved_models/pipeline_bundle.pkl',
    #model_dir='saved_models',
    #threshold=0.6
#)

# Simpan hasil
#df_out.to_csv('data/predictions.csv', index=False)
